# 22 - Publication Regulatory Panels

This notebook builds publication-style panels for human-vs-rest differential accessibility and differential expression, then ranks SCENIC+ TF-enhancer-gene triplets using concordant DA and DE signals.

Scope:
- DA volcano plots for Enterocytes, Macrophages, Naive B cells, Plasma B cells, and T cells
- DE volcano plots for the same cell types
- Triplet ranking using `-log10(padj) * log2FoldChange` for enhancer DA, TF DE, and target DE
- Concordant-sign filtering across all three layers
- Summary scatter plots linking enhancer DA to target-gene DE
- Genome-track plots for the top 10 human-up triplets per cell type

Note: the cell-type-specific SCENIC+ export in `human_specific_eregulon_links_by_celltype.tsv` currently carries unmapped `peak_id = -1`, so this notebook falls back to the mapped global human triplets in `Human_eRegulons_with_peak_ids.tsv` and scores them separately for each cell type.

In [4]:
from __future__ import annotations



import math

import re

import sys

from pathlib import Path



import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

import pyBigWig

import seaborn as sns

from IPython.display import display



PROJECT_ROOT = Path('/home/jjanssens/jjans/analysis/adult_intestine/peaks/peak_calling/atac_pipeline')

if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(0, str(PROJECT_ROOT))



from src.utils import resolve_path

from src.visualization import get_gtf_features, smooth_signal



sns.set_theme(style='whitegrid', context='talk')

plt.rcParams['pdf.fonttype'] = 42

plt.rcParams['ps.fonttype'] = 42

plt.rcParams['figure.dpi'] = 120



CONSENSUS_ROOT = Path('/home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3')

RNA_ROOT = Path('/home/jjanssens/jjans/analysis/adult_intestine/rna/pseudobulk_deseq2')

GENE_SET_DIR = Path('/home/jjanssens/jjans/analysis/adult_intestine/rna/integration/gene_sets')

ANNO_PATH = CONSENSUS_ROOT / '07_master_annotation' / 'master_annotation_final.tsv'

ATAC_RESULTS_DIR = CONSENSUS_ROOT / '13_deseq2_R_pseudobulk' / 'differential_results' / 'evolutionary_branches'

RNA_RESULTS_DIR = RNA_ROOT / 'rna_differential_results'

RNA_META_PATH = RNA_ROOT / 'pseudobulk_meta_all_species.csv'

SCENIC_DIR = CONSENSUS_ROOT / '16_scenicplus_eregulon_divergence'

SCENIC_GLOBAL_PATH = SCENIC_DIR / 'Human_eRegulons_with_peak_ids.tsv'

SCENIC_CELLTYPE_PATH = SCENIC_DIR / 'human_specific_eregulon_links_by_celltype.tsv'

BW_FILTER_BASE = Path(resolve_path('/cluster/home/jjanssens/jjans/analysis/adult_intestine/dl_models/enterocytes_v0/bigwigs_filter'))

GTF_PATH = Path('/home/jjanssens/jjans/analysis/cerebellum/genomes_new/homo_sapiens/gencode.v48.annotation.gtf.gz')



OUT_DIR = CONSENSUS_ROOT / '22_publication_regulatory_panels'

FIG_DIR = OUT_DIR / 'figures'

TABLE_DIR = OUT_DIR / 'tables'

TRACK_DIR = OUT_DIR / 'triplet_tracks'

GENE_SET_FIG_DIR = OUT_DIR / 'gene_set_triplet_figures'

GENE_SET_HEATMAP_DIR = OUT_DIR / 'gene_set_expression_heatmaps'

for path in (OUT_DIR, FIG_DIR, TABLE_DIR, TRACK_DIR, GENE_SET_FIG_DIR, GENE_SET_HEATMAP_DIR):

    path.mkdir(parents=True, exist_ok=True)



CELL_TYPES = ['Enterocytes', 'Macrophages', 'Naive B cells', 'Plasma B cells', 'T cells']

CONTRAST = 'Div_Human_vs_AllPrimates'

PADJ_THRESH = 0.05

LFC_THRESH = 1.0

TOP_LABELS = 6

TOP_TRIPLETS_TO_PLOT = 10

REGION_PADDING = 3000

COLLAPSE_DISTANCE_BP = 10000

SPECIES_ORDER = ['Human', 'Chimpanzee', 'Bonobo', 'Gorilla', 'Macaque', 'Marmoset']

SPECIES_COLORS = {

    'Human': '#1f9d55',

    'Chimpanzee': '#f28e2b',

    'Bonobo': '#8f63c9',

    'Gorilla': '#4e79a7',

    'Macaque': '#e15759',

    'Marmoset': '#76b7b2',

}

VOLCANO_COLORS = {

    'Human-high': '#c23b22',

    'Rest-high': '#2563eb',

    'Not significant': '#c7c7c7',

}



print('Python executable:', sys.executable)

print('Notebook output root:', OUT_DIR)

print('BigWig discovery root exists:', BW_FILTER_BASE.exists())

print('Gene-set directory exists:', GENE_SET_DIR.exists())

Python executable: /local1/scratch/jjans/miniforge3/envs/scenicplus_scenicplus10a2_py311/bin/python
Notebook output root: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels
BigWig discovery root exists: True
Gene-set directory exists: True


/local1/scratch/jjans/miniforge3/envs/scenicplus_scenicplus10a2_py311/lib/python3.11/site-packages/pycisTopic/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [5]:
def celltype_to_dirname(cell_type: str) -> str:

    return re.sub(r'[^A-Za-z0-9]', '.', cell_type)





def output_slug(value: str) -> str:

    return re.sub(r'[^A-Za-z0-9]+', '_', str(value)).strip('_').lower()





def sanitize_token(value: str) -> str:

    token = re.sub(r'[^A-Za-z0-9._-]+', '_', str(value)).strip('_')

    return token or 'item'





def normalize_label(value: str) -> str:

    return re.sub(r'[^a-z0-9]+', '', str(value).lower())





def pick_first(existing: list[str], candidates: list[str]) -> str | None:

    for candidate in candidates:

        if candidate in existing:

            return candidate

    return None





def component_score(log2fc: pd.Series, padj: pd.Series) -> pd.Series:

    padj_safe = padj.clip(lower=1e-300, upper=1.0)

    return -np.log10(padj_safe) * log2fc





def significance_metric(*padj_series: pd.Series) -> pd.Series:

    metric = None

    for padj in padj_series:

        current = -np.log10(pd.Series(padj).clip(lower=1e-300, upper=1.0))

        metric = current if metric is None else metric * current

    return metric if metric is not None else pd.Series(dtype=float)





def save_figure(fig: plt.Figure, out_base: Path, dpi: int = 300) -> tuple[Path, Path]:

    out_base.parent.mkdir(parents=True, exist_ok=True)

    pdf_path = out_base.with_suffix('.pdf')

    png_path = out_base.with_suffix('.png')

    fig.savefig(pdf_path, bbox_inches='tight')

    fig.savefig(png_path, dpi=dpi, bbox_inches='tight')

    plt.close(fig)

    return pdf_path, png_path





def style_axes(ax: plt.Axes, xlabel: str, ylabel: str) -> None:

    ax.set_xlabel(xlabel, fontsize=17, fontweight='bold')

    ax.set_ylabel(ylabel, fontsize=17, fontweight='bold')

    ax.tick_params(axis='both', labelsize=13)





def annotate_points(

    ax: plt.Axes,

    label_df: pd.DataFrame,

    x_col: str,

    y_col: str,

    text_col: str,

    fontsize: int = 9,

) -> None:

    for idx, (_, row) in enumerate(label_df.iterrows()):

        x_offset = 10 if row[x_col] >= 0 else -10

        y_offset = 6 + (idx % 5) * 4

        ax.annotate(

            str(row[text_col]),

            xy=(row[x_col], row[y_col]),

            xytext=(x_offset, y_offset),

            textcoords='offset points',

            fontsize=fontsize,

            fontstyle='italic',

            ha='left' if row[x_col] >= 0 else 'right',

            va='bottom',

            arrowprops={'arrowstyle': '-', 'lw': 0.6, 'color': '#666666', 'alpha': 0.65},

        )





def normalize_point_sizes(metric: pd.Series, size_min: float = 30, size_max: float = 220) -> pd.Series:

    metric = pd.Series(metric).fillna(0.0).clip(lower=0.0)

    if metric.empty:

        return metric

    max_value = float(metric.max())

    if max_value <= 0:

        return pd.Series(size_min, index=metric.index, dtype=float)

    scaled = np.sqrt(metric / max_value)

    return pd.Series(size_min + scaled * (size_max - size_min), index=metric.index, dtype=float)





def load_annotation(annotation_path: Path) -> pd.DataFrame:

    annotation = pd.read_csv(annotation_path, sep='\t', low_memory=False)

    columns = annotation.columns.tolist()

    gene_col = pick_first(columns, ['Human_gene', 'closest_gene', 'gene'])

    dist_col = pick_first(columns, ['Human_gene_dist', 'distance_to_gene', 'gene_distance'])

    chrom_col = pick_first(columns, ['Human_chr', 'chr', 'Chromosome'])

    start_col = pick_first(columns, ['Human_start', 'start', 'Start'])

    end_col = pick_first(columns, ['Human_end', 'end', 'End'])



    keep = ['peak_id']

    for column in [gene_col, dist_col, chrom_col, start_col, end_col]:

        if column is not None and column not in keep:

            keep.append(column)



    out = annotation[keep].copy()

    rename_map = {}

    if gene_col is not None:

        rename_map[gene_col] = 'nearest_gene'

    if dist_col is not None:

        rename_map[dist_col] = 'gene_distance'

    if chrom_col is not None:

        rename_map[chrom_col] = 'chrom'

    if start_col is not None:

        rename_map[start_col] = 'start'

    if end_col is not None:

        rename_map[end_col] = 'end'

    out = out.rename(columns=rename_map)



    for column in ['nearest_gene', 'gene_distance', 'chrom', 'start', 'end']:

        if column not in out.columns:

            out[column] = pd.NA



    out['nearest_gene'] = out['nearest_gene'].replace({'.': pd.NA, '': pd.NA}).astype('string')

    return out.drop_duplicates(subset=['peak_id'])





def load_atac_result(cell_type: str, contrast: str = CONTRAST) -> pd.DataFrame:

    path = ATAC_RESULTS_DIR / celltype_to_dirname(cell_type) / f'{contrast}.csv'

    df = pd.read_csv(path)

    id_col = 'peak_id' if 'peak_id' in df.columns else df.columns[0]

    df = df.rename(columns={id_col: 'peak_id'}).copy()

    df['cell_type'] = cell_type

    return df





def load_rna_result(cell_type: str, contrast: str = CONTRAST) -> pd.DataFrame:

    path = RNA_RESULTS_DIR / celltype_to_dirname(cell_type) / f'{contrast}_RNA_DE.csv'

    df = pd.read_csv(path)

    gene_col = 'gene' if 'gene' in df.columns else df.columns[0]

    df = df.rename(columns={gene_col: 'gene'}).copy()

    df['cell_type'] = cell_type

    df['gene_key'] = df['gene'].astype(str).str.upper()

    return df





def load_scenic_triplets(celltype_path: Path, global_path: Path) -> tuple[str, pd.DataFrame]:

    if celltype_path.exists():

        cell_df = pd.read_csv(celltype_path, sep='\t')

        if 'peak_id' in cell_df.columns and (cell_df['peak_id'].astype(str) != '-1').any():

            keep_cols = [column for column in ['TF', 'Region', 'Gene', 'regulation', 'peak_id', 'cell_type'] if column in cell_df.columns]

            out = cell_df[keep_cols].dropna(subset=['peak_id']).copy()

            out = out[out['peak_id'].astype(str) != '-1']

            out['TF_key'] = out['TF'].astype(str).str.upper()

            out['Gene_key'] = out['Gene'].astype(str).str.upper()

            return 'cell_type_specific', out.drop_duplicates()



    usecols = ['TF', 'Region', 'Gene', 'regulation', 'peak_id', 'species', 'triplet_rank']

    out = pd.read_csv(global_path, sep='\t', usecols=usecols)

    out = out[out['species'] == 'Human'].copy()

    out = out.dropna(subset=['peak_id'])

    out = out[out['peak_id'].astype(str) != '-1']

    out['TF_key'] = out['TF'].astype(str).str.upper()

    out['Gene_key'] = out['Gene'].astype(str).str.upper()

    out = out.drop_duplicates(subset=['TF', 'peak_id', 'Gene', 'regulation'])

    return 'global_human_triplets', out





def discover_bigwigs(base_dir: Path) -> dict[str, dict[str, str]]:

    discovered: dict[str, dict[str, str]] = {}

    if not base_dir.exists():

        return discovered



    for ct_dir in sorted(base_dir.iterdir()):

        if not ct_dir.is_dir():

            continue

        paths = {}

        for species in SPECIES_ORDER:

            candidate = ct_dir / f'{species.lower()}_{ct_dir.name}.hg38.cov.bw'

            if candidate.exists():

                paths[species] = str(candidate)

        if len(paths) == len(SPECIES_ORDER):

            discovered[ct_dir.name] = paths

    return discovered





def match_bigwig_cell_type(cell_type: str, available: dict[str, dict[str, str]]) -> str | None:

    aliases = {

        'Enterocytes': ['enterocytes'],

        'Macrophages': ['macrophages'],

        'Naive B cells': ['naivebcells', 'naive_b_cells'],

        'Plasma B cells': ['plasmabcells', 'plasma_b_cells'],

        'T cells': ['tcells', 't_cells'],

    }

    normalized_to_folder = {normalize_label(folder): folder for folder in available}

    targets = [normalize_label(cell_type), normalize_label(celltype_to_dirname(cell_type))]

    targets.extend(normalize_label(alias) for alias in aliases.get(cell_type, []))

    for target in targets:

        if target in normalized_to_folder:

            return normalized_to_folder[target]

    return None





def peak_to_region(annotation_df: pd.DataFrame, peak_id: str, pad: int = REGION_PADDING) -> str | None:

    hit = annotation_df.loc[annotation_df['peak_id'] == peak_id, ['chrom', 'start', 'end']]

    if hit.empty:

        return None

    row = hit.iloc[0]

    if pd.isna(row['chrom']) or pd.isna(row['start']) or pd.isna(row['end']):

        return None

    start = max(0, int(row['start']) - pad)

    end = int(row['end']) + pad

    return f"{row['chrom']}:{start}-{end}"





def parse_region(region: str) -> tuple[str, int, int]:

    chrom, coords = region.split(':', 1)

    start_str, end_str = coords.replace(',', '').split('-', 1)

    return chrom, int(start_str), int(end_str)





def load_gene_sets(gene_set_dir: Path) -> dict[str, set[str]]:

    gene_sets: dict[str, set[str]] = {}

    for path in sorted(gene_set_dir.glob('*.txt')):

        if path.stem.lower() == 'readme':

            continue

        genes: set[str] = set()

        for line in path.read_text().splitlines():

            line = line.strip()

            if not line or line.startswith('#'):

                continue

            for token in re.split(r'[\s,;]+', line):

                token = token.strip()

                if token:

                    genes.add(token.upper())

        if genes:

            gene_sets[path.stem] = genes

    return gene_sets





def load_species_counts(species: str, count_cache: dict[str, pd.DataFrame]) -> pd.DataFrame:

    if species not in count_cache:

        count_cache[species] = pd.read_parquet(RNA_ROOT / species / 'pseudobulk_counts.parquet')

    return count_cache[species]





def build_species_expression_matrix(

    cell_type: str,

    genes: list[str],

    rna_meta: pd.DataFrame,

    count_cache: dict[str, pd.DataFrame],

) -> pd.DataFrame:

    ordered_genes: list[str] = []

    seen: set[str] = set()

    for gene in genes:

        gene_name = str(gene)

        gene_key = gene_name.upper()

        if gene_key in seen:

            continue

        seen.add(gene_key)

        ordered_genes.append(gene_name)



    if not ordered_genes:

        return pd.DataFrame(columns=SPECIES_ORDER)



    species_vectors: dict[str, pd.Series] = {}

    for species in SPECIES_ORDER:

        meta_sub = rna_meta[(rna_meta['cell_type'] == cell_type) & (rna_meta['species'] == species)].copy()

        if meta_sub.empty:

            continue

        counts = load_species_counts(species, count_cache)

        sample_ids = [sample_id for sample_id in meta_sub['sample_id'].tolist() if sample_id in counts.columns]

        if not sample_ids:

            continue



        upper_to_actual = pd.Series(counts.index.astype(str), index=pd.Index(counts.index.astype(str)).str.upper())

        upper_to_actual = upper_to_actual[~upper_to_actual.index.duplicated(keep='first')]



        subset_rows = []

        subset_labels = []

        for gene in ordered_genes:

            actual = upper_to_actual.get(gene.upper())

            if actual is None:

                continue

            subset_rows.append(actual)

            subset_labels.append(gene)



        if not subset_rows:

            continue



        subset = counts.loc[subset_rows, sample_ids].astype(float)

        subset.index = subset_labels

        library_sizes = counts[sample_ids].sum(axis=0).replace(0, np.nan)

        log_cpm = np.log2(subset.div(library_sizes, axis=1).multiply(1e6) + 1.0)

        species_vectors[species] = log_cpm.mean(axis=1)



    if not species_vectors:

        return pd.DataFrame(index=ordered_genes, columns=SPECIES_ORDER, dtype=float)



    return pd.DataFrame(species_vectors).reindex(index=ordered_genes, columns=SPECIES_ORDER)





def zscore_rows(df: pd.DataFrame) -> pd.DataFrame:

    centered = df.sub(df.mean(axis=1), axis=0)

    scale = df.std(axis=1).replace(0, np.nan)

    out = centered.div(scale, axis=0).clip(-2.5, 2.5)

    return out.fillna(0.0)





def prepare_volcano_frame(df: pd.DataFrame, label_column: str) -> pd.DataFrame:

    plot_df = df.dropna(subset=['padj', 'log2FoldChange']).copy()

    plot_df['padj_safe'] = plot_df['padj'].clip(lower=1e-300)

    plot_df['neg_log10_padj'] = -np.log10(plot_df['padj_safe'])

    plot_df['rank_score'] = plot_df['log2FoldChange'].abs() * plot_df['neg_log10_padj']

    plot_df['direction'] = 'Not significant'

    plot_df.loc[(plot_df['padj'] < PADJ_THRESH) & (plot_df['log2FoldChange'] >= LFC_THRESH), 'direction'] = 'Human-high'

    plot_df.loc[(plot_df['padj'] < PADJ_THRESH) & (plot_df['log2FoldChange'] <= -LFC_THRESH), 'direction'] = 'Rest-high'

    fallback = plot_df.get('peak_id', pd.Series(index=plot_df.index, dtype='object'))

    plot_df['label'] = plot_df[label_column].fillna(fallback).astype(str)

    invalid = plot_df['label'].isin(['<NA>', 'nan', '.', ''])

    plot_df.loc[invalid, 'label'] = fallback.loc[invalid].astype(str)

    return plot_df





def select_top_labels(plot_df: pd.DataFrame, n_labels: int = TOP_LABELS) -> pd.DataFrame:

    label_frames = []

    for direction in ['Human-high', 'Rest-high']:

        sub = plot_df[plot_df['direction'] == direction].sort_values(['rank_score', 'baseMean'], ascending=[False, False]).copy()

        if sub.empty:

            continue

        sub = sub.drop_duplicates(subset=['label'])

        label_frames.append(sub.head(n_labels))

    if not label_frames:

        return plot_df.iloc[0:0].copy()

    return pd.concat(label_frames, ignore_index=False)





def plot_volcano_for_cell_type(plot_df: pd.DataFrame, cell_type: str, out_base: Path) -> dict[str, object]:

    x_max = max(2.0, float(np.nanquantile(np.abs(plot_df['log2FoldChange']), 0.995)) * 1.08)

    y_max = max(5.0, float(np.nanquantile(plot_df['neg_log10_padj'], 0.995)) * 1.08)



    fig, ax = plt.subplots(figsize=(7.2, 6.2))

    for direction, color in VOLCANO_COLORS.items():

        sub = plot_df[plot_df['direction'] == direction]

        ax.scatter(sub['log2FoldChange'], sub['neg_log10_padj'], s=16, alpha=0.68, c=color, edgecolors='none')



    ax.axvline(LFC_THRESH, linestyle='--', color='#999999', linewidth=0.9)

    ax.axvline(-LFC_THRESH, linestyle='--', color='#999999', linewidth=0.9)

    ax.axhline(-math.log10(PADJ_THRESH), linestyle='--', color='#999999', linewidth=0.9)

    ax.set_xlim(-x_max, x_max)

    ax.set_ylim(0, y_max)

    style_axes(ax, 'log2 fold-change', '-log10 adjusted p-value')



    label_df = select_top_labels(plot_df)

    annotate_points(ax, label_df, x_col='log2FoldChange', y_col='neg_log10_padj', text_col='label')



    handles = [

        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=VOLCANO_COLORS[key], markersize=8, label=key)

        for key in ['Human-high', 'Rest-high', 'Not significant']

    ]

    ax.legend(handles=handles, loc='upper right', frameon=False, fontsize=11)

    sns.despine(ax=ax)



    pdf_path, png_path = save_figure(fig, out_base)

    return {

        'cell_type': cell_type,

        'n_total': len(plot_df),

        'n_human_high': int((plot_df['direction'] == 'Human-high').sum()),

        'n_rest_high': int((plot_df['direction'] == 'Rest-high').sum()),

        'pdf_path': str(pdf_path),

        'png_path': str(png_path),

    }





def prepare_triplet_label_df(df: pd.DataFrame, n_labels: int = 6) -> pd.DataFrame:

    if df.empty:

        return df.iloc[0:0].copy()

    label_df = df.sort_values(['total_score_abs', 'sig_metric', 'triplet_rank'], ascending=[False, False, True]).copy()

    label_df = label_df.drop_duplicates(subset=['TF', 'Gene'])

    label_df['triplet_label'] = label_df['TF'].astype(str) + ' -> ' + label_df['Gene'].astype(str)

    return label_df.head(n_labels)





def plot_triplet_scatter_for_cell_type(

    all_df: pd.DataFrame,

    selected_df: pd.DataFrame,

    label_df: pd.DataFrame,

    out_base: Path,

) -> dict[str, object]:

    plot_df = all_df.copy()

    if len(plot_df) > 18000:

        plot_df = plot_df.nlargest(18000, 'total_score_abs').copy()

    if not selected_df.empty:

        plot_df = pd.concat([plot_df, selected_df], ignore_index=False).drop_duplicates()



    fig, ax = plt.subplots(figsize=(7.4, 6.3))

    ax.scatter(plot_df['da_lfc'], plot_df['target_lfc'], s=18, c='#d4d4d4', alpha=0.35, edgecolors='none')



    color_limit = max(1.5, float(np.nanquantile(np.abs(all_df['tf_lfc']), 0.98))) if not all_df.empty else 2.0

    if not selected_df.empty:

        scatter = ax.scatter(

            selected_df['da_lfc'],

            selected_df['target_lfc'],

            c=selected_df['tf_lfc'],

            cmap='coolwarm',

            vmin=-color_limit,

            vmax=color_limit,

            s=normalize_point_sizes(selected_df['sig_metric']),

            alpha=0.88,

            edgecolors='white',

            linewidths=0.35,

        )

        cbar = fig.colorbar(scatter, ax=ax, pad=0.02)

        cbar.set_label('TF log2 fold-change', fontsize=14)

        cbar.ax.tick_params(labelsize=12)



    if not label_df.empty:

        ax.scatter(

            label_df['da_lfc'],

            label_df['target_lfc'],

            s=normalize_point_sizes(label_df['sig_metric']) + 40,

            facecolors='none',

            edgecolors='black',

            linewidths=1.1,

        )

        annotate_points(ax, label_df, x_col='da_lfc', y_col='target_lfc', text_col='triplet_label', fontsize=8)



    ax.axhline(0, color='#9a9a9a', linewidth=0.9)

    ax.axvline(0, color='#9a9a9a', linewidth=0.9)

    style_axes(ax, 'Enhancer log2 fold-change', 'Target gene log2 fold-change')

    sns.despine(ax=ax)



    pdf_path, png_path = save_figure(fig, out_base)

    return {

        'n_background': len(plot_df),

        'n_selected': len(selected_df),

        'n_labeled': len(label_df),

        'pdf_path': str(pdf_path),

        'png_path': str(png_path),

    }





def select_nonredundant_triplets(

    df: pd.DataFrame,

    annotation_df: pd.DataFrame,

    n: int = TOP_TRIPLETS_TO_PLOT,

    distance_bp: int = COLLAPSE_DISTANCE_BP,

) -> pd.DataFrame:

    if df.empty:

        return df.iloc[0:0].copy()



    coords = annotation_df[['peak_id', 'chrom', 'start', 'end']].dropna(subset=['peak_id', 'chrom', 'start', 'end']).copy()

    coords['midpoint'] = ((coords['start'].astype(int) + coords['end'].astype(int)) / 2).astype(int)

    coords = coords.drop_duplicates(subset=['peak_id']).set_index('peak_id')



    selected_indices: list[int] = []

    occupied: list[tuple[str, int]] = []

    ordered = df.sort_values(

        ['core_sig', 'triplet_sig', 'total_score_abs', 'sig_metric', 'triplet_rank'],

        ascending=[False, False, False, False, True],

    )



    for idx, row in ordered.iterrows():

        peak_id = row['peak_id']

        if peak_id not in coords.index:

            continue

        coord = coords.loc[peak_id]

        midpoint = int(coord['midpoint'])

        chrom = str(coord['chrom'])

        if any(chrom == used_chrom and abs(midpoint - used_midpoint) <= distance_bp for used_chrom, used_midpoint in occupied):

            continue

        selected_indices.append(idx)

        occupied.append((chrom, midpoint))

        if len(selected_indices) >= n:

            break



    return ordered.loc[selected_indices].copy()





def extract_bigwig_signal(bw_path: str, chrom: str, start: int, end: int, smooth_window: int = 25) -> np.ndarray:

    with pyBigWig.open(bw_path) as bw:

        bw_chrom = chrom if chrom in bw.chroms() else f'chr{chrom}' if f'chr{chrom}' in bw.chroms() else chrom

        values = np.nan_to_num(np.asarray(bw.values(bw_chrom, start, end)))

    if smooth_window and len(values) >= smooth_window:

        values = smooth_signal(values, smooth_window)

    return values





def plot_gene_models(ax: plt.Axes, chrom: str, start: int, end: int) -> None:

    if not GTF_PATH.exists():

        ax.axis('off')

        return

    features = get_gtf_features(str(GTF_PATH), chrom, start, end)

    if features.empty or 'gene_name' not in features.columns:

        ax.axis('off')

        return



    genes = (

        features.dropna(subset=['gene_name'])

        .groupby('gene_name', as_index=False)

        .agg(start=('start', 'min'), end=('end', 'max'))

        .sort_values('start')

    )

    if genes.empty:

        ax.axis('off')

        return



    last_end_by_level: list[int] = []

    for _, row in genes.iterrows():

        level = 0

        while level < len(last_end_by_level) and int(row['start']) <= last_end_by_level[level]:

            level += 1

        if level == len(last_end_by_level):

            last_end_by_level.append(int(row['end']))

        else:

            last_end_by_level[level] = int(row['end'])

        y_pos = -level

        ax.plot([row['start'], row['end']], [y_pos, y_pos], color='#222222', linewidth=2.0, solid_capstyle='round')

        ax.text((int(row['start']) + int(row['end'])) / 2, y_pos + 0.12, row['gene_name'], fontsize=8, ha='center', va='bottom')



    ax.set_xlim(start, end)

    ax.set_ylim(-len(last_end_by_level) - 0.5, 1.0)

    ax.set_yticks([])

    ax.tick_params(axis='x', labelsize=10)

    sns.despine(ax=ax, left=True, top=True, right=True)





def plot_normalized_accessibility_tracks(

    bw_paths: dict[str, str],

    region: str,

    out_base: Path,

    smooth_window: int = 25,

) -> tuple[Path, Path]:

    chrom, start, end = parse_region(region)

    x = np.arange(start, end)



    signal_map = {

        species: extract_bigwig_signal(bw_paths[species], chrom, start, end, smooth_window=smooth_window)

        for species in SPECIES_ORDER

    }

    human_max = float(np.nanmax(signal_map['Human'])) if len(signal_map['Human']) else 0.0

    scale_max = human_max if human_max > 0 else max(float(np.nanmax(values)) for values in signal_map.values())

    scale_max = max(scale_max, 1e-6)



    fig, axes = plt.subplots(

        len(SPECIES_ORDER) + 1,

        1,

        figsize=(12.0, len(SPECIES_ORDER) * 1.2 + 2.1),

        sharex=True,

        gridspec_kw={'height_ratios': [1.0] * len(SPECIES_ORDER) + [0.75]},

    )



    for ax, species in zip(axes[:-1], SPECIES_ORDER):

        scaled = np.clip(signal_map[species] / scale_max, a_min=0.0, a_max=None)

        ax.fill_between(x, scaled, color=SPECIES_COLORS[species], alpha=0.92, linewidth=0)

        ax.plot(x, scaled, color=SPECIES_COLORS[species], linewidth=0.8)

        ax.set_ylim(0, 1.05)

        ax.set_yticks([0, 1])

        ax.set_ylabel(species, rotation=0, ha='right', va='center', fontsize=10, labelpad=22)

        ax.tick_params(axis='y', labelsize=9)

        ax.tick_params(axis='x', bottom=False, labelbottom=False)

        sns.despine(ax=ax, top=True, right=True, bottom=True)



    plot_gene_models(axes[-1], chrom, start, end)

    axes[-1].set_xlabel(f'{chrom}:{start:,}-{end:,}', fontsize=12, fontweight='bold')



    pdf_path, png_path = save_figure(fig, out_base)

    return pdf_path, png_path





def plot_species_expression_heatmap(expr_df: pd.DataFrame, out_base: Path) -> tuple[Path, Path, pd.DataFrame]:

    if expr_df.empty:

        raise ValueError('Expression matrix is empty.')

    plot_df = zscore_rows(expr_df.fillna(0.0))

    fig_height = min(max(4.8, plot_df.shape[0] * 0.22 + 1.3), 28)

    fig, ax = plt.subplots(figsize=(7.4, fig_height))

    sns.heatmap(

        plot_df,

        cmap='RdBu_r',

        center=0,

        vmin=-2.5,

        vmax=2.5,

        linewidths=0.15,

        linecolor='white',

        cbar_kws={'label': 'z-scored log2 CPM'},

        ax=ax,

    )

    style_axes(ax, 'Species', 'Genes')

    ax.tick_params(axis='x', labelrotation=45, labelsize=12)

    ax.tick_params(axis='y', labelsize=6 if plot_df.shape[0] <= 50 else 5 if plot_df.shape[0] <= 100 else 4)

    plt.setp(ax.get_xticklabels(), ha='right')



    pdf_path, png_path = save_figure(fig, out_base)

    return pdf_path, png_path, plot_df


In [12]:
from matplotlib.lines import Line2D

import anndata as ad



ATLAS_H5AD = Path('/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/rna/integration/scvi_integration/nhp_atlas_merged_scvi_input_20250520_112026.h5ad')

atlas_adata = globals().get('atlas_adata', None)

atlas_obs_cache = globals().get('atlas_obs_cache', None)

atlas_var_lookup = globals().get('atlas_var_lookup', None)

triplet_dotplot_cache = globals().get('triplet_dotplot_cache', {})





def save_figure(fig, out_base: Path, dpi: int = 300, tight: bool = True) -> tuple[Path, Path]:

    out_base.parent.mkdir(parents=True, exist_ok=True)

    pdf_path = out_base.with_suffix('.pdf')

    png_path = out_base.with_suffix('.png')

    if tight:

        fig.savefig(pdf_path, bbox_inches='tight')

        fig.savefig(png_path, dpi=dpi, bbox_inches='tight')

    else:

        fig.savefig(pdf_path)

        fig.savefig(png_path, dpi=dpi)

    plt.close(fig)

    return pdf_path, png_path





def place_non_overlapping_annotations(

    ax: plt.Axes,

    label_df: pd.DataFrame,

    x_col: str,

    y_col: str,

    text_col: str,

    fontsize: int = 9,

    expand: tuple[float, float] = (1.04, 1.18),

) -> pd.DataFrame:

    if label_df.empty:

        return label_df.iloc[0:0].copy()



    fig = ax.figure

    fig.canvas.draw()

    accepted_indices: list[object] = []

    placed_boxes = []

    renderer = fig.canvas.get_renderer()



    for idx, (_, row) in enumerate(label_df.iterrows()):

        magnitude = 8 + (idx % 4) * 2

        candidates = [

            (magnitude, magnitude, 'left', 'bottom'),

            (-magnitude, magnitude, 'right', 'bottom'),

            (magnitude, -magnitude, 'left', 'top'),

            (-magnitude, -magnitude, 'right', 'top'),

            (0, magnitude + 4, 'center', 'bottom'),

        ]

        accepted = False

        for dx, dy, ha, va in candidates:

            annotation = ax.annotate(

                str(row[text_col]),

                xy=(row[x_col], row[y_col]),

                xytext=(dx, dy),

                textcoords='offset points',

                fontsize=fontsize,

                fontstyle='italic',

                ha=ha,

                va=va,

                arrowprops={'arrowstyle': '-', 'lw': 0.55, 'color': '#666666', 'alpha': 0.65},

            )

            fig.canvas.draw()

            bbox = annotation.get_window_extent(renderer=renderer).expanded(*expand)

            if any(bbox.overlaps(other) for other in placed_boxes):

                annotation.remove()

                continue

            placed_boxes.append(bbox)

            accepted_indices.append(label_df.index[idx])

            accepted = True

            break

        if not accepted:

            continue



    return label_df.loc[accepted_indices].copy()





def build_volcano_axes(ax: plt.Axes, plot_df: pd.DataFrame) -> None:

    x_max = max(2.0, float(np.nanquantile(np.abs(plot_df['log2FoldChange']), 0.995)) * 1.08)

    y_max = max(5.0, float(np.nanquantile(plot_df['neg_log10_padj'], 0.995)) * 1.08)

    for direction, color in VOLCANO_COLORS.items():

        sub = plot_df[plot_df['direction'] == direction]

        ax.scatter(sub['log2FoldChange'], sub['neg_log10_padj'], s=16, alpha=0.68, c=color, edgecolors='none')

    ax.axvline(LFC_THRESH, linestyle='--', color='#999999', linewidth=0.9)

    ax.axvline(-LFC_THRESH, linestyle='--', color='#999999', linewidth=0.9)

    ax.axhline(-math.log10(PADJ_THRESH), linestyle='--', color='#999999', linewidth=0.9)

    ax.set_xlim(-x_max, x_max)

    ax.set_ylim(0, y_max)

    style_axes(ax, 'log2 fold-change', '-log10 adjusted p-value')

    handles = [

        Line2D([0], [0], marker='o', color='w', markerfacecolor=VOLCANO_COLORS[key], markersize=8, label=key)

        for key in ['Human-high', 'Rest-high', 'Not significant']

    ]

    ax.legend(handles=handles, loc='upper right', frameon=False, fontsize=11)

    sns.despine(ax=ax)





def plot_volcano_for_cell_type(plot_df: pd.DataFrame, cell_type: str, out_base: Path) -> dict[str, object]:

    requested_labels = select_top_labels(plot_df)



    fig, ax = plt.subplots(figsize=(7.2, 6.2))

    build_volcano_axes(ax, plot_df)

    pdf_no_labels, png_no_labels = save_figure(fig, out_base.parent / f'{out_base.name}__no_labels')



    fig, ax = plt.subplots(figsize=(7.2, 6.2))

    build_volcano_axes(ax, plot_df)

    placed_labels = place_non_overlapping_annotations(

        ax,

        requested_labels,

        x_col='log2FoldChange',

        y_col='neg_log10_padj',

        text_col='label',

        fontsize=8,

    )

    pdf_labeled, png_labeled = save_figure(fig, out_base.parent / f'{out_base.name}__labeled')



    return {

        'cell_type': cell_type,

        'n_total': len(plot_df),

        'n_human_high': int((plot_df['direction'] == 'Human-high').sum()),

        'n_rest_high': int((plot_df['direction'] == 'Rest-high').sum()),

        'n_requested_labels': len(requested_labels),

        'n_placed_labels': len(placed_labels),

        'pdf_path': str(pdf_labeled),

        'png_path': str(png_labeled),

        'pdf_path_no_labels': str(pdf_no_labels),

        'png_path_no_labels': str(png_no_labels),

    }





def build_triplet_scatter_axes(ax: plt.Axes, fig, all_df: pd.DataFrame, selected_df: pd.DataFrame):

    plot_df = all_df.copy()

    if len(plot_df) > 18000:

        plot_df = plot_df.nlargest(18000, 'total_score_abs').copy()

    if not selected_df.empty:

        plot_df = pd.concat([plot_df, selected_df], ignore_index=False).drop_duplicates()



    ax.scatter(plot_df['da_lfc'], plot_df['target_lfc'], s=18, c='#d4d4d4', alpha=0.35, edgecolors='none')

    color_limit = max(1.5, float(np.nanquantile(np.abs(all_df['tf_lfc']), 0.98))) if not all_df.empty else 2.0

    scatter = None

    if not selected_df.empty:

        scatter = ax.scatter(

            selected_df['da_lfc'],

            selected_df['target_lfc'],

            c=selected_df['tf_lfc'],

            cmap='coolwarm',

            vmin=-color_limit,

            vmax=color_limit,

            s=normalize_point_sizes(selected_df['sig_metric']),

            alpha=0.88,

            edgecolors='white',

            linewidths=0.35,

        )

        cbar = fig.colorbar(scatter, ax=ax, pad=0.02)

        cbar.set_label('TF log2 fold-change', fontsize=14)

        cbar.ax.tick_params(labelsize=12)



    ax.axhline(0, color='#9a9a9a', linewidth=0.9)

    ax.axvline(0, color='#9a9a9a', linewidth=0.9)

    style_axes(ax, 'Enhancer log2 fold-change', 'Target gene log2 fold-change')

    sns.despine(ax=ax)

    return plot_df





def plot_triplet_scatter_for_cell_type(

    all_df: pd.DataFrame,

    selected_df: pd.DataFrame,

    label_df: pd.DataFrame,

    out_base: Path,

) -> dict[str, object]:

    fig, ax = plt.subplots(figsize=(7.4, 6.3))

    plot_df = build_triplet_scatter_axes(ax, fig, all_df, selected_df)

    pdf_no_labels, png_no_labels = save_figure(fig, out_base.parent / f'{out_base.name}__no_labels')



    fig, ax = plt.subplots(figsize=(7.4, 6.3))

    plot_df = build_triplet_scatter_axes(ax, fig, all_df, selected_df)

    if not label_df.empty:

        ax.scatter(

            label_df['da_lfc'],

            label_df['target_lfc'],

            s=normalize_point_sizes(label_df['sig_metric']) + 40,

            facecolors='none',

            edgecolors='black',

            linewidths=1.1,

        )

    placed_labels = place_non_overlapping_annotations(

        ax,

        label_df,

        x_col='da_lfc',

        y_col='target_lfc',

        text_col='triplet_label',

        fontsize=8,

    )

    pdf_labeled, png_labeled = save_figure(fig, out_base.parent / f'{out_base.name}__labeled')



    return {

        'n_background': len(plot_df),

        'n_selected': len(selected_df),

        'n_requested_labels': len(label_df),

        'n_placed_labels': len(placed_labels),

        'pdf_path': str(pdf_labeled),

        'png_path': str(png_labeled),

        'pdf_path_no_labels': str(pdf_no_labels),

        'png_path_no_labels': str(png_no_labels),

        'placed_label_df': placed_labels,

    }





def get_atlas_adata():

    global atlas_adata

    if atlas_adata is None:

        atlas_adata = ad.read_h5ad(ATLAS_H5AD, backed='r')

    return atlas_adata





def get_atlas_obs() -> pd.DataFrame:

    global atlas_obs_cache

    if atlas_obs_cache is None:

        atlas_obs_cache = get_atlas_adata().obs[['Age', 'ct', 'Species']].copy()

        atlas_obs_cache['Age'] = atlas_obs_cache['Age'].astype(str)

        atlas_obs_cache['ct'] = atlas_obs_cache['ct'].astype(str)

        atlas_obs_cache['Species'] = atlas_obs_cache['Species'].astype(str)

    return atlas_obs_cache





def get_atlas_var_lookup() -> pd.Series:

    global atlas_var_lookup

    if atlas_var_lookup is None:

        var_names = pd.Index(get_atlas_adata().var_names.astype(str))

        atlas_var_lookup = pd.Series(var_names, index=var_names.str.upper())

        atlas_var_lookup = atlas_var_lookup[~atlas_var_lookup.index.duplicated(keep='first')]

    return atlas_var_lookup





def compute_triplet_dotplot_stats(cell_type: str, genes: list[str]) -> pd.DataFrame:

    ordered_genes = []

    seen = set()

    for gene in genes:

        gene_name = str(gene)

        gene_key = gene_name.upper()

        if gene_key in seen:

            continue

        seen.add(gene_key)

        ordered_genes.append(gene_name)



    cache_key = (cell_type, tuple(g.upper() for g in ordered_genes))

    if cache_key in triplet_dotplot_cache:

        return triplet_dotplot_cache[cache_key].copy()



    if not ordered_genes:

        out = pd.DataFrame(columns=['cell_type', 'species', 'gene_label', 'avg_expr', 'pct_expr'])

        triplet_dotplot_cache[cache_key] = out

        return out.copy()



    obs = get_atlas_obs()

    mask = (obs['Age'] == 'Adult') & (obs['ct'] == cell_type) & (obs['Species'].isin(SPECIES_ORDER))

    if not mask.any():

        out = pd.DataFrame(columns=['cell_type', 'species', 'gene_label', 'avg_expr', 'pct_expr'])

        triplet_dotplot_cache[cache_key] = out

        return out.copy()



    lookup = get_atlas_var_lookup()

    actual_genes = []

    display_genes = []

    for gene_name in ordered_genes:

        actual = lookup.get(gene_name.upper())

        if actual is None:

            continue

        actual_genes.append(actual)

        display_genes.append(gene_name)



    if not actual_genes:

        out = pd.DataFrame(columns=['cell_type', 'species', 'gene_label', 'avg_expr', 'pct_expr'])

        triplet_dotplot_cache[cache_key] = out

        return out.copy()



    subset = get_atlas_adata()[mask.to_numpy(), actual_genes]

    species_values = obs.loc[mask, 'Species'].to_numpy()

    matrix = subset.X

    matrix = matrix.toarray() if hasattr(matrix, 'toarray') else np.asarray(matrix)

    if matrix.ndim == 1:

        matrix = matrix[:, None]



    rows = []

    for gene_idx, gene_name in enumerate(display_genes):

        values = matrix[:, gene_idx]

        for species_name in SPECIES_ORDER:

            species_mask = species_values == species_name

            if species_mask.sum() == 0:

                avg_expr = np.nan

                pct_expr = 0.0

            else:

                species_values_gene = np.asarray(values[species_mask], dtype=float)

                avg_expr = float(np.mean(np.log1p(species_values_gene)))

                pct_expr = float(np.mean(species_values_gene > 0) * 100.0)

            rows.append({

                'cell_type': cell_type,

                'species': species_name,

                'gene_label': gene_name,

                'avg_expr': avg_expr,

                'pct_expr': pct_expr,

            })



    out = pd.DataFrame(rows)

    triplet_dotplot_cache[cache_key] = out

    return out.copy()





def plot_triplet_dotplot(ax: plt.Axes, dotplot_df: pd.DataFrame, tf_name: str, gene_name: str):

    if dotplot_df.empty:

        ax.text(0.5, 0.5, 'No atlas expression available', ha='center', va='center', fontsize=11)

        ax.axis('off')

        return None



    row_order = [f'TF: {tf_name}', f'Target: {gene_name}']

    display_map = {tf_name: f'TF: {tf_name}', gene_name: f'Target: {gene_name}'}

    plot_df = dotplot_df.copy()

    plot_df['row_label'] = plot_df['gene_label'].map(display_map).fillna(plot_df['gene_label'])

    plot_df = plot_df[plot_df['row_label'].isin(row_order)].copy()

    plot_df['x'] = plot_df['species'].map({species: idx for idx, species in enumerate(SPECIES_ORDER)})

    plot_df['y'] = plot_df['row_label'].map({row_order[0]: 1, row_order[1]: 0})

    plot_df['size'] = 40 + (plot_df['pct_expr'].clip(lower=0, upper=100) / 100.0) * 380



    scatter = ax.scatter(

        plot_df['x'],

        plot_df['y'],

        s=plot_df['size'],

        c=plot_df['avg_expr'],

        cmap='YlOrRd',

        edgecolors='#333333',

        linewidths=0.45,

    )

    ax.set_xticks(range(len(SPECIES_ORDER)))

    ax.set_xticklabels(SPECIES_ORDER, rotation=40, ha='right', fontsize=10)

    ax.set_yticks([1, 0])

    ax.set_yticklabels(row_order, fontsize=11)

    ax.set_xlim(-0.5, len(SPECIES_ORDER) - 0.5)

    ax.set_ylim(-0.6, 1.6)

    ax.grid(axis='x', color='#ececec', linewidth=0.7)

    ax.grid(axis='y', color='#f2f2f2', linewidth=0.7)

    sns.despine(ax=ax, top=True, right=True)



    for pct in [25, 50, 75]:

        ax.scatter([], [], s=40 + (pct / 100.0) * 380, c='white', edgecolors='#333333', linewidths=0.45, label=f'{pct}%')

    ax.legend(title='% expressing', loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=9, title_fontsize=10)

    return scatter





def plot_gene_models(ax: plt.Axes, chrom: str, start: int, end: int, max_labels: int = 12) -> None:

    if not GTF_PATH.exists():

        ax.axis('off')

        return

    features = get_gtf_features(str(GTF_PATH), chrom, start, end)

    if features.empty or 'gene_name' not in features.columns:

        ax.axis('off')

        return



    genes = (

        features.dropna(subset=['gene_name'])

        .groupby('gene_name', as_index=False)

        .agg(start=('start', 'min'), end=('end', 'max'))

        .sort_values('start')

        .head(max_labels)

    )

    if genes.empty:

        ax.axis('off')

        return



    last_end_by_level: list[int] = []

    for _, row in genes.iterrows():

        level = 0

        while level < len(last_end_by_level) and int(row['start']) <= last_end_by_level[level]:

            level += 1

        if level == len(last_end_by_level):

            last_end_by_level.append(int(row['end']))

        else:

            last_end_by_level[level] = int(row['end'])

        y_pos = -level

        ax.plot([row['start'], row['end']], [y_pos, y_pos], color='#222222', linewidth=2.0, solid_capstyle='round')

        ax.text((int(row['start']) + int(row['end'])) / 2, y_pos + 0.12, row['gene_name'], fontsize=8, ha='center', va='bottom', clip_on=True)



    ax.set_xlim(start, end)

    ax.set_ylim(-len(last_end_by_level) - 0.5, 1.0)

    ax.set_yticks([])

    ax.tick_params(axis='x', labelsize=10)

    sns.despine(ax=ax, left=True, top=True, right=True)





def plot_normalized_accessibility_tracks(

    bw_paths: dict[str, str],

    region: str,

    out_base: Path,

    smooth_window: int = 25,

) -> tuple[Path, Path]:

    chrom, start, end = parse_region(region)

    x = np.arange(start, end)

    signal_map = {species: extract_bigwig_signal(bw_paths[species], chrom, start, end, smooth_window=smooth_window) for species in SPECIES_ORDER}

    human_max = float(np.nanmax(signal_map['Human'])) if len(signal_map['Human']) else 0.0

    scale_max = human_max if human_max > 0 else max(float(np.nanmax(values)) for values in signal_map.values())

    scale_max = max(scale_max, 1e-6)



    fig, axes = plt.subplots(len(SPECIES_ORDER) + 1, 1, figsize=(12.0, len(SPECIES_ORDER) * 1.2 + 2.1), sharex=True, gridspec_kw={'height_ratios': [1.0] * len(SPECIES_ORDER) + [0.75]})

    for ax, species in zip(axes[:-1], SPECIES_ORDER):

        scaled = np.clip(signal_map[species] / scale_max, a_min=0.0, a_max=None)

        ax.fill_between(x, scaled, color=SPECIES_COLORS[species], alpha=0.92, linewidth=0)

        ax.plot(x, scaled, color=SPECIES_COLORS[species], linewidth=0.8)

        ax.set_ylim(0, 1.05)

        ax.set_yticks([0, 1])

        ax.set_ylabel(species, rotation=0, ha='right', va='center', fontsize=10, labelpad=22)

        ax.tick_params(axis='y', labelsize=9)

        ax.tick_params(axis='x', bottom=False, labelbottom=False)

        sns.despine(ax=ax, top=True, right=True, bottom=True)

    plot_gene_models(axes[-1], chrom, start, end)

    axes[-1].set_xlabel(f'{chrom}:{start:,}-{end:,}', fontsize=12, fontweight='bold')

    fig.subplots_adjust(hspace=0.16)

    return save_figure(fig, out_base, tight=False)





def plot_triplet_slide_panel(

    bw_paths: dict[str, str],

    region: str,

    cell_type: str,

    tf_name: str,

    gene_name: str,

    out_base: Path,

    smooth_window: int = 25,

) -> tuple[Path, Path, pd.DataFrame]:

    dotplot_df = compute_triplet_dotplot_stats(cell_type, [tf_name, gene_name])

    chrom, start, end = parse_region(region)

    x = np.arange(start, end)

    signal_map = {species: extract_bigwig_signal(bw_paths[species], chrom, start, end, smooth_window=smooth_window) for species in SPECIES_ORDER}

    human_max = float(np.nanmax(signal_map['Human'])) if len(signal_map['Human']) else 0.0

    scale_max = human_max if human_max > 0 else max(float(np.nanmax(values)) for values in signal_map.values())

    scale_max = max(scale_max, 1e-6)



    fig = plt.figure(figsize=(12.2, len(SPECIES_ORDER) * 1.0 + 3.7))

    gs = fig.add_gridspec(len(SPECIES_ORDER) + 2, 1, height_ratios=[1.25] + [1.0] * len(SPECIES_ORDER) + [0.75], hspace=0.16)

    ax_dot = fig.add_subplot(gs[0, 0])

    scatter = plot_triplet_dotplot(ax_dot, dotplot_df, tf_name=tf_name, gene_name=gene_name)

    if scatter is not None:

        cbar = fig.colorbar(scatter, ax=ax_dot, pad=0.02, fraction=0.04)

        cbar.set_label('Mean log1p expression', fontsize=11)

        cbar.ax.tick_params(labelsize=9)



    for row_idx, species in enumerate(SPECIES_ORDER, start=1):

        ax = fig.add_subplot(gs[row_idx, 0])

        scaled = np.clip(signal_map[species] / scale_max, a_min=0.0, a_max=None)

        ax.fill_between(x, scaled, color=SPECIES_COLORS[species], alpha=0.92, linewidth=0)

        ax.plot(x, scaled, color=SPECIES_COLORS[species], linewidth=0.8)

        ax.set_ylim(0, 1.05)

        ax.set_yticks([0, 1])

        ax.set_ylabel(species, rotation=0, ha='right', va='center', fontsize=10, labelpad=22)

        ax.tick_params(axis='y', labelsize=9)

        ax.tick_params(axis='x', bottom=False, labelbottom=False)

        sns.despine(ax=ax, top=True, right=True, bottom=True)



    ax_gene = fig.add_subplot(gs[len(SPECIES_ORDER) + 1, 0])

    plot_gene_models(ax_gene, chrom, start, end)

    ax_gene.set_xlabel(f'{chrom}:{start:,}-{end:,}', fontsize=12, fontweight='bold')

    fig.subplots_adjust(left=0.12, right=0.9)

    pdf_path, png_path = save_figure(fig, out_base, tight=False)

    return pdf_path, png_path, dotplot_df


In [23]:
from matplotlib.lines import Line2D

ULTRA_ROBUST_COLOR = '#f59e0b'
FOCUS_TRIPLET_COLOR = '#d97706'
SPECIAL_TRIPLET_TARGETS = {
    'Enterocytes': 'ALPI',
    'Plasma B cells': 'AREG',
}
ATAC_ULTRA_ROBUST_ROOT = CONSENSUS_ROOT / '13_deseq2_R_pseudobulk'


def load_atac_ultra_robust_peak_ids(cell_type: str, contrast: str = CONTRAST) -> set[str]:
    ultra_path = ATAC_ULTRA_ROBUST_ROOT / celltype_to_dirname(cell_type) / 'atac_ultra_robust' / f'{contrast}_DA.csv'
    if not ultra_path.exists():
        return set()

    ultra_df = pd.read_csv(ultra_path)
    if 'peak_id' not in ultra_df.columns:
        return set()

    return set(ultra_df['peak_id'].dropna().astype(str))


def _draw_volcano_panel(
    plot_df: pd.DataFrame,
    label_df: pd.DataFrame,
    out_base: Path,
    ultra_robust_peak_ids: set[str] | None = None,
) -> tuple[Path, Path, int]:
    ultra_robust_peak_ids = ultra_robust_peak_ids or set()
    has_peak_ids = 'peak_id' in plot_df.columns
    highlight_mask = plot_df['peak_id'].astype(str).isin(ultra_robust_peak_ids) if has_peak_ids else pd.Series(False, index=plot_df.index)

    abs_lfc = np.abs(plot_df['log2FoldChange'].to_numpy()) if not plot_df.empty else np.array([0.0])
    neg_log10 = plot_df['neg_log10_padj'].to_numpy() if not plot_df.empty else np.array([0.0])
    x_max = max(2.4, float(np.nanquantile(abs_lfc, 0.998)) * 1.18 + 0.15)
    y_max = max(5.5, float(np.nanquantile(neg_log10, 0.998)) * 1.16 + 0.35)

    fig, ax = plt.subplots(figsize=(7.8, 6.8))
    fig.subplots_adjust(left=0.14, right=0.98, bottom=0.14, top=0.80)

    for direction, color in VOLCANO_COLORS.items():
        sub = plot_df[(plot_df['direction'] == direction) & (~highlight_mask)]
        if sub.empty:
            continue
        ax.scatter(
            sub['log2FoldChange'],
            sub['neg_log10_padj'],
            s=18,
            alpha=0.64,
            c=color,
            edgecolors='none',
            rasterized=len(sub) > 5000,
        )

    n_highlighted = int(highlight_mask.sum())
    if n_highlighted:
        highlighted = plot_df.loc[highlight_mask]
        ax.scatter(
            highlighted['log2FoldChange'],
            highlighted['neg_log10_padj'],
            s=28,
            alpha=0.96,
            c=ULTRA_ROBUST_COLOR,
            edgecolors='black',
            linewidths=0.28,
            zorder=4,
        )

    ax.axvline(LFC_THRESH, linestyle='--', color='#999999', linewidth=0.9)
    ax.axvline(-LFC_THRESH, linestyle='--', color='#999999', linewidth=0.9)
    ax.axhline(-math.log10(PADJ_THRESH), linestyle='--', color='#999999', linewidth=0.9)
    ax.set_xlim(-x_max, x_max)
    ax.set_ylim(0, y_max)
    style_axes(ax, 'log2 fold-change', '-log10 adjusted p-value')

    if not label_df.empty:
        annotate_points(ax, label_df, x_col='log2FoldChange', y_col='neg_log10_padj', text_col='label')

    handles = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=VOLCANO_COLORS[key], markersize=8, label=key)
        for key in ['Human-high', 'Rest-high', 'Not significant']
    ]
    if n_highlighted:
        handles.append(
            Line2D([0], [0], marker='o', color='w', markerfacecolor=ULTRA_ROBUST_COLOR, markeredgecolor='black', markersize=8, label='Ultra-robust peaks')
        )

    ax.legend(
        handles=handles,
        loc='lower center',
        bbox_to_anchor=(0.5, 1.03),
        ncol=min(len(handles), 4),
        frameon=False,
        fontsize=11,
        handletextpad=0.45,
        columnspacing=1.1,
    )
    sns.despine(ax=ax)

    pdf_path, png_path = save_figure(fig, out_base)
    return pdf_path, png_path, n_highlighted


def plot_volcano_for_cell_type(
    plot_df: pd.DataFrame,
    cell_type: str,
    out_base: Path,
    ultra_robust_peak_ids: set[str] | None = None,
) -> dict[str, object]:
    label_df = select_top_labels(plot_df)
    pdf_path, png_path, _ = _draw_volcano_panel(plot_df, label_df, out_base)

    summary = {
        'cell_type': cell_type,
        'n_total': len(plot_df),
        'n_human_high': int((plot_df['direction'] == 'Human-high').sum()),
        'n_rest_high': int((plot_df['direction'] == 'Rest-high').sum()),
        'pdf_path': str(pdf_path),
        'png_path': str(png_path),
        'n_ultra_robust': 0,
        'pdf_path_ultra_robust': None,
        'png_path_ultra_robust': None,
    }

    if ultra_robust_peak_ids and 'peak_id' in plot_df.columns:
        ultra_base = out_base.parent / f'{out_base.name}__ultra_robust'
        ultra_pdf_path, ultra_png_path, n_highlighted = _draw_volcano_panel(
            plot_df,
            label_df,
            ultra_base,
            ultra_robust_peak_ids=ultra_robust_peak_ids,
        )
        summary['n_ultra_robust'] = n_highlighted
        summary['pdf_path_ultra_robust'] = str(ultra_pdf_path)
        summary['png_path_ultra_robust'] = str(ultra_png_path)

    return summary


def select_focus_triplet_for_target(df: pd.DataFrame, target_gene: str | None) -> pd.DataFrame:
    if df.empty or not target_gene:
        return df.iloc[0:0].copy()

    target_key = normalize_label(target_gene)
    focus_df = df[df['Gene'].astype(str).map(normalize_label) == target_key].copy()
    if focus_df.empty:
        return focus_df

    focus_df = focus_df.sort_values(
        ['core_sig', 'triplet_sig', 'total_score_abs', 'sig_metric', 'triplet_rank'],
        ascending=[False, False, False, False, True],
    ).head(1).copy()
    focus_df['triplet_label'] = focus_df['TF'].astype(str) + ' -> ' + focus_df['Gene'].astype(str)
    return focus_df


def _draw_triplet_scatter_panel(
    all_df: pd.DataFrame,
    selected_df: pd.DataFrame,
    label_df: pd.DataFrame,
    focus_df: pd.DataFrame,
    out_base: Path,
) -> tuple[Path, Path]:
    plot_df = all_df.copy()
    if len(plot_df) > 18000:
        plot_df = plot_df.nlargest(18000, 'total_score_abs').copy()
    if not selected_df.empty:
        plot_df = pd.concat([plot_df, selected_df], ignore_index=False).drop_duplicates()
    if not focus_df.empty:
        plot_df = pd.concat([plot_df, focus_df], ignore_index=False).drop_duplicates()

    fig, ax = plt.subplots(figsize=(7.6, 6.5))
    fig.subplots_adjust(left=0.14, right=0.90, bottom=0.14, top=0.96)
    ax.scatter(plot_df['da_lfc'], plot_df['target_lfc'], s=18, c='#d4d4d4', alpha=0.35, edgecolors='none')

    scatter = None
    color_limit = max(1.5, float(np.nanquantile(np.abs(all_df['tf_lfc']), 0.98))) if not all_df.empty else 2.0
    if not selected_df.empty:
        scatter = ax.scatter(
            selected_df['da_lfc'],
            selected_df['target_lfc'],
            c=selected_df['tf_lfc'],
            cmap='coolwarm',
            vmin=-color_limit,
            vmax=color_limit,
            s=normalize_point_sizes(selected_df['sig_metric']),
            alpha=0.88,
            edgecolors='white',
            linewidths=0.35,
        )

    if not focus_df.empty:
        focus_sizes = normalize_point_sizes(focus_df['sig_metric']) + 120
        ax.scatter(
            focus_df['da_lfc'],
            focus_df['target_lfc'],
            s=focus_sizes,
            c=FOCUS_TRIPLET_COLOR,
            alpha=0.97,
            edgecolors='black',
            linewidths=0.9,
            zorder=5,
        )
        ax.scatter(
            focus_df['da_lfc'],
            focus_df['target_lfc'],
            s=focus_sizes + 80,
            facecolors='none',
            edgecolors='black',
            linewidths=1.1,
            zorder=6,
        )

    annotated_df = label_df.copy()
    if not focus_df.empty:
        annotated_df = pd.concat([annotated_df, focus_df], ignore_index=False)
        annotated_df = annotated_df.drop_duplicates(subset=['TF', 'Gene', 'peak_id'])

    if not annotated_df.empty:
        ax.scatter(
            annotated_df['da_lfc'],
            annotated_df['target_lfc'],
            s=normalize_point_sizes(annotated_df['sig_metric']) + 40,
            facecolors='none',
            edgecolors='black',
            linewidths=1.1,
            zorder=7,
        )
        annotate_points(ax, annotated_df, x_col='da_lfc', y_col='target_lfc', text_col='triplet_label', fontsize=8)

    if scatter is not None:
        cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
        cbar.set_label('TF log2 fold-change', fontsize=14)
        cbar.ax.tick_params(labelsize=12)

    ax.axhline(0, color='#9a9a9a', linewidth=0.9)
    ax.axvline(0, color='#9a9a9a', linewidth=0.9)
    style_axes(ax, 'Enhancer log2 fold-change', 'Target gene log2 fold-change')
    sns.despine(ax=ax)

    pdf_path, png_path = save_figure(fig, out_base)
    return pdf_path, png_path


def plot_triplet_scatter_for_cell_type(
    all_df: pd.DataFrame,
    selected_df: pd.DataFrame,
    label_df: pd.DataFrame,
    out_base: Path,
    focus_df: pd.DataFrame | None = None,
) -> dict[str, object]:
    focus_df = focus_df if focus_df is not None else all_df.iloc[0:0].copy()
    label_df = label_df.copy()
    if 'triplet_label' not in label_df.columns and not label_df.empty:
        label_df['triplet_label'] = label_df['TF'].astype(str) + ' -> ' + label_df['Gene'].astype(str)

    pdf_path_no_labels, png_path_no_labels = _draw_triplet_scatter_panel(
        all_df=all_df,
        selected_df=selected_df,
        label_df=all_df.iloc[0:0].copy(),
        focus_df=focus_df,
        out_base=out_base.parent / f'{out_base.name}__no_labels',
    )
    pdf_path, png_path = _draw_triplet_scatter_panel(
        all_df=all_df,
        selected_df=selected_df,
        label_df=label_df,
        focus_df=focus_df,
        out_base=out_base,
    )

    placed_label_df = label_df.copy()
    if not focus_df.empty:
        placed_label_df = pd.concat([placed_label_df, focus_df], ignore_index=False)
        placed_label_df = placed_label_df.drop_duplicates(subset=['TF', 'Gene', 'peak_id'])

    return {
        'n_background': len(all_df),
        'n_selected': len(selected_df),
        'n_requested_labels': len(label_df),
        'n_placed_labels': len(placed_label_df),
        'placed_label_df': placed_label_df.copy(),
        'pdf_path': str(pdf_path),
        'png_path': str(png_path),
        'pdf_path_no_labels': str(pdf_path_no_labels),
        'png_path_no_labels': str(png_path_no_labels),
    }

In [27]:
def _draw_volcano_panel(
    plot_df: pd.DataFrame,
    label_df: pd.DataFrame,
    out_base: Path,
    ultra_robust_peak_ids: set[str] | None = None,
) -> tuple[Path, Path, int]:
    ultra_robust_peak_ids = ultra_robust_peak_ids or set()
    has_peak_ids = 'peak_id' in plot_df.columns
    highlight_mask = plot_df['peak_id'].astype(str).isin(ultra_robust_peak_ids) if has_peak_ids else pd.Series(False, index=plot_df.index)

    abs_lfc = np.abs(plot_df['log2FoldChange'].to_numpy()) if not plot_df.empty else np.array([0.0])
    neg_log10 = plot_df['neg_log10_padj'].to_numpy() if not plot_df.empty else np.array([0.0])
    x_max = max(2.4, float(np.nanquantile(abs_lfc, 0.998)) * 1.18 + 0.15)
    y_max = max(5.5, float(np.nanquantile(neg_log10, 0.998)) * 1.16 + 0.35)

    fig, ax = plt.subplots(figsize=(7.8, 6.8))
    fig.subplots_adjust(left=0.14, right=0.98, bottom=0.14, top=0.80)

    for direction, color in VOLCANO_COLORS.items():
        sub = plot_df[(plot_df['direction'] == direction) & (~highlight_mask)]
        if sub.empty:
            continue
        ax.scatter(
            sub['log2FoldChange'],
            sub['neg_log10_padj'],
            s=18,
            alpha=0.64,
            c=color,
            edgecolors='none',
            rasterized=len(sub) > 5000,
        )

    n_highlighted = int(highlight_mask.sum())
    if n_highlighted:
        highlighted = plot_df.loc[highlight_mask]
        ax.scatter(
            highlighted['log2FoldChange'],
            highlighted['neg_log10_padj'],
            s=32,
            facecolors='none',
            edgecolors=ULTRA_ROBUST_COLOR,
            linewidths=0.65,
            alpha=0.95,
            zorder=4,
        )

    ax.axvline(LFC_THRESH, linestyle='--', color='#999999', linewidth=0.9)
    ax.axvline(-LFC_THRESH, linestyle='--', color='#999999', linewidth=0.9)
    ax.axhline(-math.log10(PADJ_THRESH), linestyle='--', color='#999999', linewidth=0.9)
    ax.set_xlim(-x_max, x_max)
    ax.set_ylim(0, y_max)
    style_axes(ax, 'log2 fold-change', '-log10 adjusted p-value')

    if not label_df.empty:
        annotate_points(ax, label_df, x_col='log2FoldChange', y_col='neg_log10_padj', text_col='label')

    handles = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=VOLCANO_COLORS[key], markersize=8, label=key)
        for key in ['Human-high', 'Rest-high', 'Not significant']
    ]
    if n_highlighted:
        handles.append(
            Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor=ULTRA_ROBUST_COLOR, markeredgewidth=1.4, markersize=8, label='Ultra-robust peaks')
        )

    ax.legend(
        handles=handles,
        loc='lower center',
        bbox_to_anchor=(0.5, 1.03),
        ncol=min(len(handles), 4),
        frameon=False,
        fontsize=11,
        handletextpad=0.45,
        columnspacing=1.1,
    )
    sns.despine(ax=ax)

    pdf_path, png_path = save_figure(fig, out_base)
    return pdf_path, png_path, n_highlighted

In [29]:
def plot_volcano_for_cell_type(
    plot_df: pd.DataFrame,
    cell_type: str,
    out_base: Path,
    ultra_robust_peak_ids: set[str] | None = None,
    n_labels: int = TOP_LABELS,
    include_labels_in_main: bool = True,
    save_labeled_variant: bool = False,
    save_no_label_variant: bool = False,
) -> dict[str, object]:
    label_df = select_top_labels(plot_df, n_labels=n_labels)
    empty_labels = plot_df.iloc[0:0].copy()
    main_label_df = label_df if include_labels_in_main else empty_labels
    pdf_path, png_path, _ = _draw_volcano_panel(plot_df, main_label_df, out_base)

    summary = {
        'cell_type': cell_type,
        'n_total': len(plot_df),
        'n_human_high': int((plot_df['direction'] == 'Human-high').sum()),
        'n_rest_high': int((plot_df['direction'] == 'Rest-high').sum()),
        'n_requested_labels': len(label_df),
        'pdf_path': str(pdf_path),
        'png_path': str(png_path),
        'pdf_path_labeled': None,
        'png_path_labeled': None,
        'pdf_path_no_labels': None,
        'png_path_no_labels': None,
        'n_ultra_robust': 0,
        'pdf_path_ultra_robust': None,
        'png_path_ultra_robust': None,
    }

    if save_labeled_variant:
        labeled_base = out_base.parent / f'{out_base.name}__labeled'
        labeled_pdf_path, labeled_png_path, _ = _draw_volcano_panel(plot_df, label_df, labeled_base)
        summary['pdf_path_labeled'] = str(labeled_pdf_path)
        summary['png_path_labeled'] = str(labeled_png_path)

    if save_no_label_variant:
        no_label_base = out_base.parent / f'{out_base.name}__no_labels'
        no_label_pdf_path, no_label_png_path, _ = _draw_volcano_panel(plot_df, empty_labels, no_label_base)
        summary['pdf_path_no_labels'] = str(no_label_pdf_path)
        summary['png_path_no_labels'] = str(no_label_png_path)

    if ultra_robust_peak_ids and 'peak_id' in plot_df.columns:
        ultra_base = out_base.parent / f'{out_base.name}__ultra_robust'
        ultra_pdf_path, ultra_png_path, n_highlighted = _draw_volcano_panel(
            plot_df,
            main_label_df,
            ultra_base,
            ultra_robust_peak_ids=ultra_robust_peak_ids,
        )
        summary['n_ultra_robust'] = n_highlighted
        summary['pdf_path_ultra_robust'] = str(ultra_pdf_path)
        summary['png_path_ultra_robust'] = str(ultra_png_path)

    return summary


def _draw_triplet_scatter_panel(
    all_df: pd.DataFrame,
    selected_df: pd.DataFrame,
    label_df: pd.DataFrame,
    focus_df: pd.DataFrame,
    out_base: Path,
    annotate_focus: bool = True,
) -> tuple[Path, Path]:
    plot_df = all_df.copy()
    if len(plot_df) > 18000:
        plot_df = plot_df.nlargest(18000, 'total_score_abs').copy()
    if not selected_df.empty:
        plot_df = pd.concat([plot_df, selected_df], ignore_index=False).drop_duplicates()
    if not focus_df.empty:
        plot_df = pd.concat([plot_df, focus_df], ignore_index=False).drop_duplicates()

    fig, ax = plt.subplots(figsize=(7.6, 6.5))
    fig.subplots_adjust(left=0.14, right=0.90, bottom=0.14, top=0.96)
    ax.scatter(plot_df['da_lfc'], plot_df['target_lfc'], s=18, c='#d4d4d4', alpha=0.35, edgecolors='none')

    scatter = None
    color_limit = max(1.5, float(np.nanquantile(np.abs(all_df['tf_lfc']), 0.98))) if not all_df.empty else 2.0
    if not selected_df.empty:
        scatter = ax.scatter(
            selected_df['da_lfc'],
            selected_df['target_lfc'],
            c=selected_df['tf_lfc'],
            cmap='coolwarm',
            vmin=-color_limit,
            vmax=color_limit,
            s=normalize_point_sizes(selected_df['sig_metric']),
            alpha=0.88,
            edgecolors='white',
            linewidths=0.35,
        )

    if not focus_df.empty:
        focus_sizes = normalize_point_sizes(focus_df['sig_metric']) + 120
        ax.scatter(
            focus_df['da_lfc'],
            focus_df['target_lfc'],
            s=focus_sizes,
            c=FOCUS_TRIPLET_COLOR,
            alpha=0.97,
            edgecolors='black',
            linewidths=0.9,
            zorder=5,
        )
        ax.scatter(
            focus_df['da_lfc'],
            focus_df['target_lfc'],
            s=focus_sizes + 80,
            facecolors='none',
            edgecolors='black',
            linewidths=1.1,
            zorder=6,
        )

    annotated_df = label_df.copy()
    if annotate_focus and not focus_df.empty:
        annotated_df = pd.concat([annotated_df, focus_df], ignore_index=False)
        annotated_df = annotated_df.drop_duplicates(subset=['TF', 'Gene', 'peak_id'])

    if not annotated_df.empty:
        ax.scatter(
            annotated_df['da_lfc'],
            annotated_df['target_lfc'],
            s=normalize_point_sizes(annotated_df['sig_metric']) + 40,
            facecolors='none',
            edgecolors='black',
            linewidths=1.1,
            zorder=7,
        )
        annotate_points(ax, annotated_df, x_col='da_lfc', y_col='target_lfc', text_col='triplet_label', fontsize=8)

    if scatter is not None:
        cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
        cbar.set_label('TF log2 fold-change', fontsize=14)
        cbar.ax.tick_params(labelsize=12)

    ax.axhline(0, color='#9a9a9a', linewidth=0.9)
    ax.axvline(0, color='#9a9a9a', linewidth=0.9)
    style_axes(ax, 'Enhancer log2 fold-change', 'Target gene log2 fold-change')
    sns.despine(ax=ax)

    pdf_path, png_path = save_figure(fig, out_base)
    return pdf_path, png_path


def plot_triplet_scatter_for_cell_type(
    all_df: pd.DataFrame,
    selected_df: pd.DataFrame,
    label_df: pd.DataFrame,
    out_base: Path,
    focus_df: pd.DataFrame | None = None,
) -> dict[str, object]:
    focus_df = focus_df if focus_df is not None else all_df.iloc[0:0].copy()
    label_df = label_df.copy()
    if 'triplet_label' not in label_df.columns and not label_df.empty:
        label_df['triplet_label'] = label_df['TF'].astype(str) + ' -> ' + label_df['Gene'].astype(str)

    empty_df = all_df.iloc[0:0].copy()

    pdf_path_no_labels, png_path_no_labels = _draw_triplet_scatter_panel(
        all_df=all_df,
        selected_df=selected_df,
        label_df=empty_df,
        focus_df=empty_df,
        out_base=out_base.parent / f'{out_base.name}__no_labels',
        annotate_focus=False,
    )

    pdf_path_focus_only = None
    png_path_focus_only = None
    if not focus_df.empty:
        pdf_path_focus_only, png_path_focus_only = _draw_triplet_scatter_panel(
            all_df=all_df,
            selected_df=selected_df,
            label_df=empty_df,
            focus_df=focus_df,
            out_base=out_base.parent / f'{out_base.name}__focus_only',
            annotate_focus=True,
        )

    pdf_path, png_path = _draw_triplet_scatter_panel(
        all_df=all_df,
        selected_df=selected_df,
        label_df=label_df,
        focus_df=focus_df,
        out_base=out_base,
        annotate_focus=True,
    )

    placed_label_df = label_df.copy()
    if not focus_df.empty:
        placed_label_df = pd.concat([placed_label_df, focus_df], ignore_index=False)
        placed_label_df = placed_label_df.drop_duplicates(subset=['TF', 'Gene', 'peak_id'])

    return {
        'n_background': len(all_df),
        'n_selected': len(selected_df),
        'n_requested_labels': len(label_df),
        'n_placed_labels': len(placed_label_df),
        'placed_label_df': placed_label_df.copy(),
        'pdf_path': str(pdf_path),
        'png_path': str(png_path),
        'pdf_path_no_labels': str(pdf_path_no_labels),
        'png_path_no_labels': str(png_path_no_labels),
        'pdf_path_focus_only': None if pdf_path_focus_only is None else str(pdf_path_focus_only),
        'png_path_focus_only': None if png_path_focus_only is None else str(png_path_focus_only),
    }

In [33]:
def plot_volcano_for_cell_type(
    plot_df: pd.DataFrame,
    cell_type: str,
    out_base: Path,
    ultra_robust_peak_ids: set[str] | None = None,
    n_labels: int = TOP_LABELS,
    include_labels_in_main: bool = True,
    save_labeled_variant: bool = False,
    save_no_label_variant: bool = False,
) -> dict[str, object]:
    abs_lfc = np.abs(plot_df['log2FoldChange'].to_numpy()) if not plot_df.empty else np.array([0.0])
    neg_log10 = plot_df['neg_log10_padj'].to_numpy() if not plot_df.empty else np.array([0.0])
    x_max = max(2.4, float(np.nanquantile(abs_lfc, 0.998)) * 1.18 + 0.15)
    y_max = max(5.5, float(np.nanquantile(neg_log10, 0.998)) * 1.16 + 0.35)

    visible_df = plot_df[
        plot_df['log2FoldChange'].abs().le(x_max) & plot_df['neg_log10_padj'].le(y_max)
    ].copy()
    label_source_df = visible_df if not visible_df.empty else plot_df
    label_df = select_top_labels(label_source_df, n_labels=n_labels)
    empty_labels = plot_df.iloc[0:0].copy()
    main_label_df = label_df if include_labels_in_main else empty_labels
    pdf_path, png_path, _ = _draw_volcano_panel(plot_df, main_label_df, out_base)

    summary = {
        'cell_type': cell_type,
        'n_total': len(plot_df),
        'n_human_high': int((plot_df['direction'] == 'Human-high').sum()),
        'n_rest_high': int((plot_df['direction'] == 'Rest-high').sum()),
        'n_requested_labels': len(label_df),
        'pdf_path': str(pdf_path),
        'png_path': str(png_path),
        'pdf_path_labeled': None,
        'png_path_labeled': None,
        'pdf_path_no_labels': None,
        'png_path_no_labels': None,
        'n_ultra_robust': 0,
        'pdf_path_ultra_robust': None,
        'png_path_ultra_robust': None,
    }

    if save_labeled_variant:
        labeled_base = out_base.parent / f'{out_base.name}__labeled'
        labeled_pdf_path, labeled_png_path, _ = _draw_volcano_panel(plot_df, label_df, labeled_base)
        summary['pdf_path_labeled'] = str(labeled_pdf_path)
        summary['png_path_labeled'] = str(labeled_png_path)

    if save_no_label_variant:
        no_label_base = out_base.parent / f'{out_base.name}__no_labels'
        no_label_pdf_path, no_label_png_path, _ = _draw_volcano_panel(plot_df, empty_labels, no_label_base)
        summary['pdf_path_no_labels'] = str(no_label_pdf_path)
        summary['png_path_no_labels'] = str(no_label_png_path)

    if ultra_robust_peak_ids and 'peak_id' in plot_df.columns:
        ultra_base = out_base.parent / f'{out_base.name}__ultra_robust'
        ultra_pdf_path, ultra_png_path, n_highlighted = _draw_volcano_panel(
            plot_df,
            main_label_df,
            ultra_base,
            ultra_robust_peak_ids=ultra_robust_peak_ids,
        )
        summary['n_ultra_robust'] = n_highlighted
        summary['pdf_path_ultra_robust'] = str(ultra_pdf_path)
        summary['png_path_ultra_robust'] = str(ultra_png_path)

    return summary

In [35]:
def _draw_triplet_scatter_panel(
    all_df: pd.DataFrame,
    selected_df: pd.DataFrame,
    label_df: pd.DataFrame,
    focus_df: pd.DataFrame,
    out_base: Path,
    annotate_focus: bool = True,
) -> tuple[Path, Path]:
    plot_df = all_df.copy()
    if len(plot_df) > 18000:
        plot_df = plot_df.nlargest(18000, 'total_score_abs').copy()
    if not selected_df.empty:
        plot_df = pd.concat([plot_df, selected_df], ignore_index=False).drop_duplicates()
    if not focus_df.empty:
        plot_df = pd.concat([plot_df, focus_df], ignore_index=False).drop_duplicates()

    fig, ax = plt.subplots(figsize=(7.6, 6.5))
    fig.subplots_adjust(left=0.14, right=0.90, bottom=0.14, top=0.96)
    ax.scatter(plot_df['da_lfc'], plot_df['target_lfc'], s=18, c='#d4d4d4', alpha=0.35, edgecolors='none')

    scatter = None
    color_limit = max(1.5, float(np.nanquantile(np.abs(all_df['tf_lfc']), 0.98))) if not all_df.empty else 2.0
    if not selected_df.empty:
        scatter = ax.scatter(
            selected_df['da_lfc'],
            selected_df['target_lfc'],
            c=selected_df['tf_lfc'],
            cmap='coolwarm',
            vmin=-color_limit,
            vmax=color_limit,
            s=normalize_point_sizes(selected_df['sig_metric']),
            alpha=0.88,
            edgecolors='white',
            linewidths=0.35,
        )

    if not focus_df.empty:
        focus_sizes = normalize_point_sizes(focus_df['sig_metric'])
        ax.scatter(
            focus_df['da_lfc'],
            focus_df['target_lfc'],
            c=focus_df['tf_lfc'],
            cmap='coolwarm',
            vmin=-color_limit,
            vmax=color_limit,
            s=focus_sizes,
            alpha=0.88,
            edgecolors='white',
            linewidths=0.35,
            zorder=5,
        )
        ax.scatter(
            focus_df['da_lfc'],
            focus_df['target_lfc'],
            s=focus_sizes,
            facecolors='none',
            edgecolors='black',
            linewidths=1.0,
            zorder=6,
        )

    annotated_df = label_df.copy()
    if annotate_focus and not focus_df.empty:
        annotated_df = pd.concat([annotated_df, focus_df], ignore_index=False)
        annotated_df = annotated_df.drop_duplicates(subset=['TF', 'Gene', 'peak_id'])

    if not annotated_df.empty:
        ax.scatter(
            annotated_df['da_lfc'],
            annotated_df['target_lfc'],
            s=normalize_point_sizes(annotated_df['sig_metric']) + 40,
            facecolors='none',
            edgecolors='black',
            linewidths=1.1,
            zorder=7,
        )
        annotate_points(ax, annotated_df, x_col='da_lfc', y_col='target_lfc', text_col='triplet_label', fontsize=8)

    if scatter is not None:
        cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
        cbar.set_label('TF log2 fold-change', fontsize=14)
        cbar.ax.tick_params(labelsize=12)

    ax.axhline(0, color='#9a9a9a', linewidth=0.9)
    ax.axvline(0, color='#9a9a9a', linewidth=0.9)
    style_axes(ax, 'Enhancer log2 fold-change', 'Target gene log2 fold-change')
    sns.despine(ax=ax)

    pdf_path, png_path = save_figure(fig, out_base)
    return pdf_path, png_path

In [37]:
def _draw_triplet_scatter_panel(
    all_df: pd.DataFrame,
    selected_df: pd.DataFrame,
    label_df: pd.DataFrame,
    focus_df: pd.DataFrame,
    out_base: Path,
    annotate_focus: bool = True,
) -> tuple[Path, Path]:
    plot_df = all_df.copy()
    if len(plot_df) > 18000:
        plot_df = plot_df.nlargest(18000, 'total_score_abs').copy()
    if not selected_df.empty:
        plot_df = pd.concat([plot_df, selected_df], ignore_index=False).drop_duplicates()
    if not focus_df.empty:
        plot_df = pd.concat([plot_df, focus_df], ignore_index=False).drop_duplicates()

    fig, ax = plt.subplots(figsize=(7.6, 6.5))
    fig.subplots_adjust(left=0.14, right=0.90, bottom=0.14, top=0.96)
    ax.scatter(plot_df['da_lfc'], plot_df['target_lfc'], s=18, c='#d4d4d4', alpha=0.35, edgecolors='none')

    scatter = None
    color_limit = max(1.5, float(np.nanquantile(np.abs(all_df['tf_lfc']), 0.98))) if not all_df.empty else 2.0
    if not selected_df.empty:
        scatter = ax.scatter(
            selected_df['da_lfc'],
            selected_df['target_lfc'],
            c=selected_df['tf_lfc'],
            cmap='coolwarm',
            vmin=-color_limit,
            vmax=color_limit,
            s=normalize_point_sizes(selected_df['sig_metric']),
            alpha=0.88,
            edgecolors='white',
            linewidths=0.35,
        )

    if not focus_df.empty:
        focus_sizes = normalize_point_sizes(focus_df['sig_metric'])
        ax.scatter(
            focus_df['da_lfc'],
            focus_df['target_lfc'],
            c=focus_df['tf_lfc'],
            cmap='coolwarm',
            vmin=-color_limit,
            vmax=color_limit,
            s=focus_sizes,
            alpha=0.88,
            edgecolors='white',
            linewidths=0.35,
            zorder=5,
        )
        ax.scatter(
            focus_df['da_lfc'],
            focus_df['target_lfc'],
            s=focus_sizes,
            facecolors='none',
            edgecolors='black',
            linewidths=1.0,
            zorder=6,
        )

    if not label_df.empty:
        ax.scatter(
            label_df['da_lfc'],
            label_df['target_lfc'],
            s=normalize_point_sizes(label_df['sig_metric']) + 40,
            facecolors='none',
            edgecolors='black',
            linewidths=1.1,
            zorder=7,
        )
        annotate_points(ax, label_df, x_col='da_lfc', y_col='target_lfc', text_col='triplet_label', fontsize=8)

    if annotate_focus and not focus_df.empty:
        annotate_points(ax, focus_df, x_col='da_lfc', y_col='target_lfc', text_col='triplet_label', fontsize=8)

    if scatter is not None:
        cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
        cbar.set_label('TF log2 fold-change', fontsize=14)
        cbar.ax.tick_params(labelsize=12)

    ax.axhline(0, color='#9a9a9a', linewidth=0.9)
    ax.axvline(0, color='#9a9a9a', linewidth=0.9)
    style_axes(ax, 'Enhancer log2 fold-change', 'Target gene log2 fold-change')
    sns.despine(ax=ax)

    pdf_path, png_path = save_figure(fig, out_base)
    return pdf_path, png_path

In [13]:
SLIDE_PANELS_PER_SIGNATURE = 3

SLIDE_PANEL_MIN_PER_CELL = 24





def build_triplet_scatter_axes(ax: plt.Axes, fig, all_df: pd.DataFrame, selected_df: pd.DataFrame):

    plot_df = all_df.copy()

    if len(plot_df) > 18000:

        plot_df = plot_df.nlargest(18000, 'total_score_abs').copy()

    if not selected_df.empty:

        plot_df = pd.concat([plot_df, selected_df], ignore_index=False).drop_duplicates()



    selected_index = pd.Index(selected_df.index)

    background_df = plot_df.loc[~plot_df.index.isin(selected_index)].copy()

    if not background_df.empty:

        ax.scatter(

            background_df['da_lfc'],

            background_df['target_lfc'],

            s=10,

            c='#dddddd',

            alpha=0.12,

            edgecolors='none',

            zorder=1,

            rasterized=len(background_df) > 5000,

        )



    color_limit = max(1.5, float(np.nanquantile(np.abs(all_df['tf_lfc']), 0.98))) if not all_df.empty else 2.0

    scatter = None

    if not selected_df.empty:

        ordered_selected = selected_df.sort_values(

            ['triplet_sig', 'core_sig', 'triplet_sig_metric', 'sig_metric', 'total_score_abs', 'triplet_rank'],

            ascending=[True, True, True, True, True, True],

        ).copy()

        scatter = ax.scatter(

            ordered_selected['da_lfc'],

            ordered_selected['target_lfc'],

            c=ordered_selected['tf_lfc'],

            cmap='coolwarm',

            vmin=-color_limit,

            vmax=color_limit,

            s=normalize_point_sizes(ordered_selected['sig_metric'], size_min=18, size_max=135),

            alpha=0.72,

            edgecolors='white',

            linewidths=0.28,

            zorder=2,

            rasterized=len(ordered_selected) > 4000,

        )

        cbar = fig.colorbar(scatter, ax=ax, pad=0.02)

        cbar.set_label('TF log2 fold-change', fontsize=14)

        cbar.ax.tick_params(labelsize=12)



    ax.axhline(0, color='#9a9a9a', linewidth=0.9)

    ax.axvline(0, color='#9a9a9a', linewidth=0.9)

    style_axes(ax, 'Enhancer log2 fold-change', 'Target gene log2 fold-change')

    sns.despine(ax=ax)

    return plot_df





def build_gene_signature_lookup(gene_sets: dict[str, set[str]]) -> dict[str, list[str]]:

    lookup: dict[str, list[str]] = {}

    for gene_set_name in sorted(gene_sets):

        for gene_key in sorted(gene_sets[gene_set_name]):

            lookup.setdefault(gene_key, []).append(gene_set_name)

    return lookup





def annotate_triplet_signatures(df: pd.DataFrame, gene_sets: dict[str, set[str]]) -> pd.DataFrame:

    out = df.copy()

    lookup = build_gene_signature_lookup(gene_sets)

    signature_order = {name: idx for idx, name in enumerate(sorted(gene_sets))}



    out['target_signatures'] = out['Gene_key'].astype(str).map(lambda key: ';'.join(lookup.get(key.upper(), [])))

    out['tf_signatures'] = out['TF_key'].astype(str).map(lambda key: ';'.join(lookup.get(key.upper(), [])))



    def choose_primary_signature(row: pd.Series) -> tuple[str, str]:

        target_hits = [value for value in str(row['target_signatures']).split(';') if value]

        tf_hits = [value for value in str(row['tf_signatures']).split(';') if value]

        if target_hits:

            return target_hits[0], 'target'

        if tf_hits:

            return tf_hits[0], 'tf'

        return 'unassigned', 'unassigned'



    primary = out.apply(choose_primary_signature, axis=1, result_type='expand')

    out['primary_signature'] = primary[0]

    out['signature_source'] = primary[1]

    out['signature_source_rank'] = out['signature_source'].map({'target': 0, 'tf': 1, 'unassigned': 2}).fillna(3).astype(int)

    out['primary_signature_rank'] = out['primary_signature'].map(signature_order).fillna(len(signature_order)).astype(int)

    out['panel_sig'] = out[['da_sig', 'target_sig', 'tf_sig']].fillna(False).any(axis=1)

    out['panel_sig_count'] = out[['da_sig', 'target_sig', 'tf_sig']].fillna(False).astype(int).sum(axis=1)

    out['panel_sig_score'] = (

        -np.log10(out['da_padj'].clip(lower=1e-300, upper=1.0)) * out['da_sig'].fillna(False).astype(int)

        + -np.log10(out['target_padj'].clip(lower=1e-300, upper=1.0)) * out['target_sig'].fillna(False).astype(int)

        + -np.log10(out['tf_padj'].clip(lower=1e-300, upper=1.0)) * out['tf_sig'].fillna(False).astype(int)

    )

    out['signature_matches'] = out.apply(

        lambda row: ';'.join([value for value in [row['target_signatures'], row['tf_signatures']] if isinstance(value, str) and value and value != 'nan']),

        axis=1,

    )

    return out





def build_panel_candidate_df(df: pd.DataFrame, gene_sets: dict[str, set[str]]) -> pd.DataFrame:

    out = annotate_triplet_signatures(df, gene_sets)

    out = out[out['panel_sig']].copy()

    out = out.sort_values(

        ['signature_source_rank', 'primary_signature_rank', 'panel_sig_count', 'triplet_sig', 'core_sig', 'panel_sig_score', 'total_score_abs', 'triplet_rank'],

        ascending=[True, True, False, False, False, False, False, True],

    )

    out['panel_rank'] = np.arange(1, len(out) + 1)

    return out





def choose_panel_render_subset(panel_df: pd.DataFrame) -> pd.DataFrame:

    if panel_df.empty:

        return panel_df.iloc[0:0].copy()

    grouped = panel_df.groupby(['signature_source', 'primary_signature'], group_keys=False, dropna=False).head(SLIDE_PANELS_PER_SIGNATURE).copy()

    grouped = grouped.sort_values('panel_rank').copy()

    min_target = min(SLIDE_PANEL_MIN_PER_CELL, len(panel_df))

    if len(grouped) < min_target:

        extra = panel_df.loc[~panel_df.index.isin(grouped.index)].head(min_target - len(grouped)).copy()

        grouped = pd.concat([grouped, extra], ignore_index=False)

    return grouped.sort_values('panel_rank').copy()





def prepare_triplet_dotplot_display(dotplot_df: pd.DataFrame, tf_name: str, gene_name: str) -> pd.DataFrame:

    if dotplot_df.empty:

        return dotplot_df.copy()

    label_map = {tf_name: 'TF', gene_name: 'Target'}

    plot_df = dotplot_df.copy()

    plot_df = plot_df[plot_df['gene_label'].isin([tf_name, gene_name])].copy()

    plot_df['feature_label'] = plot_df['gene_label'].map(label_map)

    plot_df['feature_index'] = plot_df['feature_label'].map({'TF': 0, 'Target': 1})

    plot_df['avg_expr_z'] = plot_df.groupby('gene_label')['avg_expr'].transform(

        lambda values: pd.Series(0.0, index=values.index) if float(values.std(ddof=0) or 0.0) == 0.0 else (values - values.mean()) / values.std(ddof=0)

    )

    plot_df['avg_expr_z'] = plot_df['avg_expr_z'].clip(-2.5, 2.5).fillna(0.0)

    plot_df['size'] = 24 + (plot_df['pct_expr'].clip(lower=0, upper=100) / 100.0) * 210

    plot_df['species_index'] = plot_df['species'].map({species: idx for idx, species in enumerate(SPECIES_ORDER)})

    return plot_df.sort_values(['species_index', 'feature_index']).copy()





def draw_triplet_dotplot_row(ax: plt.Axes, row_df: pd.DataFrame, species: str, show_headers: bool = False):

    ax.axhspan(-0.5, 0.5, color='#f7f7f7', zorder=0)

    scatter = None

    if not row_df.empty:

        scatter = ax.scatter(

            row_df['feature_index'],

            np.zeros(len(row_df)),

            s=row_df['size'],

            c=row_df['avg_expr_z'],

            cmap='RdBu_r',

            vmin=-2.5,

            vmax=2.5,

            edgecolors='#333333',

            linewidths=0.4,

            zorder=2,

        )

    else:

        ax.text(0.5, 0.0, 'NA', ha='center', va='center', fontsize=9, color='#666666')



    ax.set_xlim(-0.6, 1.6)

    ax.set_ylim(-0.55, 0.55)

    ax.set_yticks([])

    ax.set_xticks([0, 1])

    if show_headers:

        ax.set_xticklabels(['TF', 'Target'], fontsize=10, fontweight='bold')

        ax.tick_params(axis='x', top=True, labeltop=True, bottom=False, labelbottom=False, length=0)

    else:

        ax.set_xticklabels(['', ''])

        ax.tick_params(axis='x', top=False, labeltop=False, bottom=False, labelbottom=False, length=0)

    ax.set_ylabel(species, rotation=0, ha='right', va='center', fontsize=10, labelpad=22)

    sns.despine(ax=ax, left=True, right=True, bottom=not show_headers, top=not show_headers)

    return scatter





def plot_triplet_slide_panel(

    bw_paths: dict[str, str],

    region: str,

    cell_type: str,

    tf_name: str,

    gene_name: str,

    out_base: Path,

    smooth_window: int = 25,

) -> tuple[Path, Path, pd.DataFrame]:

    dotplot_df = compute_triplet_dotplot_stats(cell_type, [tf_name, gene_name])

    dotplot_display_df = prepare_triplet_dotplot_display(dotplot_df, tf_name=tf_name, gene_name=gene_name)

    chrom, start, end = parse_region(region)

    x = np.arange(start, end)

    signal_map = {species: extract_bigwig_signal(bw_paths[species], chrom, start, end, smooth_window=smooth_window) for species in SPECIES_ORDER}

    human_max = float(np.nanmax(signal_map['Human'])) if len(signal_map['Human']) else 0.0

    scale_max = human_max if human_max > 0 else max(float(np.nanmax(values)) for values in signal_map.values())

    scale_max = max(scale_max, 1e-6)



    fig = plt.figure(figsize=(12.8, len(SPECIES_ORDER) * 1.0 + 2.4))

    gs = fig.add_gridspec(

        len(SPECIES_ORDER) + 1,

        2,

        width_ratios=[1.35, 8.0],

        height_ratios=[1.0] * len(SPECIES_ORDER) + [0.75],

        hspace=0.16,

        wspace=0.08,

    )



    scatter_handle = None

    track_axes = []

    for row_idx, species in enumerate(SPECIES_ORDER):

        ax_dot = fig.add_subplot(gs[row_idx, 0])

        species_df = dotplot_display_df[dotplot_display_df['species'] == species].copy()

        scatter_handle = draw_triplet_dotplot_row(ax_dot, species_df, species=species, show_headers=row_idx == 0) or scatter_handle



        ax_track = fig.add_subplot(gs[row_idx, 1], sharex=track_axes[0] if track_axes else None)

        track_axes.append(ax_track)

        scaled = np.clip(signal_map[species] / scale_max, a_min=0.0, a_max=None)

        ax_track.fill_between(x, scaled, color=SPECIES_COLORS[species], alpha=0.92, linewidth=0)

        ax_track.plot(x, scaled, color=SPECIES_COLORS[species], linewidth=0.8)

        ax_track.set_ylim(0, 1.05)

        ax_track.set_yticks([0, 1])

        ax_track.tick_params(axis='y', labelsize=8)

        ax_track.tick_params(axis='x', bottom=False, labelbottom=False)

        sns.despine(ax=ax_track, top=True, right=True, bottom=True)



    ax_blank = fig.add_subplot(gs[len(SPECIES_ORDER), 0])

    ax_blank.axis('off')

    ax_gene = fig.add_subplot(gs[len(SPECIES_ORDER), 1], sharex=track_axes[0] if track_axes else None)

    plot_gene_models(ax_gene, chrom, start, end)

    ax_gene.set_xlabel(f'{chrom}:{start:,}-{end:,}', fontsize=12, fontweight='bold')



    fig.subplots_adjust(left=0.12, right=0.98)

    pdf_path, png_path = save_figure(fig, out_base, tight=False)

    return pdf_path, png_path, dotplot_display_df


In [7]:
annotation_df = load_annotation(ANNO_PATH)

atac_results = {cell_type: load_atac_result(cell_type) for cell_type in CELL_TYPES}

rna_results = {cell_type: load_rna_result(cell_type) for cell_type in CELL_TYPES}

scenic_source, scenic_triplets = load_scenic_triplets(SCENIC_CELLTYPE_PATH, SCENIC_GLOBAL_PATH)

available_bigwigs = discover_bigwigs(BW_FILTER_BASE)

rna_meta = pd.read_csv(RNA_META_PATH)

gene_sets = load_gene_sets(GENE_SET_DIR)

species_count_cache: dict[str, pd.DataFrame] = {}



summary_rows = []

for cell_type in CELL_TYPES:

    atac_df = atac_results[cell_type]

    rna_df = rna_results[cell_type]

    summary_rows.append({

        'cell_type': cell_type,

        'atac_rows': len(atac_df),

        'rna_rows': len(rna_df),

        'atac_sig': int(((atac_df['padj'] < PADJ_THRESH) & (atac_df['log2FoldChange'].abs() >= LFC_THRESH)).sum()),

        'rna_sig': int(((rna_df['padj'] < PADJ_THRESH) & (rna_df['log2FoldChange'].abs() >= LFC_THRESH)).sum()),

        'bigwig_folder': match_bigwig_cell_type(cell_type, available_bigwigs),

    })



summary_df = pd.DataFrame(summary_rows)

gene_set_summary_df = pd.DataFrame([

    {'gene_set': set_name, 'n_genes': len(genes)}

    for set_name, genes in sorted(gene_sets.items())

])



print('SCENIC+ source:', scenic_source)

print('Triplets available:', len(scenic_triplets))

print('Available bigwig folders with all species:', len(available_bigwigs))

print('Gene sets loaded:', len(gene_sets))

display(summary_df)

display(gene_set_summary_df.head(10))

display(scenic_triplets.head())

SCENIC+ source: global_human_triplets
Triplets available: 114262
Available bigwig folders with all species: 26
Gene sets loaded: 15


,cell_type,atac_rows,rna_rows,atac_sig,rna_sig,bigwig_folder
0,Enterocytes,366077,11414,30654,2466,enterocytes
1,Macrophages,131522,11168,21709,1349,macrophages
2,Naive B cells,89714,10890,4891,832,naive_b_cells
3,Plasma B cells,137409,11108,22819,1868,plasma_b_cells
4,T cells,230533,11489,23480,1900,t_cells


,gene_set,n_genes
0,bmp_tgfb,165
1,cytokines_chemokines,304
2,ecm_adhesion,415
3,enteroendocrine,61
4,enteroendocrine_fetal,32
5,epithelial_barrier,172
6,epithelial_barrier_manual,61
7,growth_factor_receptors,80
8,growth_factors,165
9,human_tfs,1639


,Region,Gene,TF,regulation,triplet_rank,peak_id,species,TF_key,Gene_key
0,chr11:34053306-34053806,CAPRIN1,AHCTF1,1,37433,unified_164327,Human,AHCTF1,CAPRIN1
1,chr16:18925566-18926066,SMG1,AHCTF1,1,90058,unified_359743,Human,AHCTF1,SMG1
2,chr10:73816-74316,ZMYND11,AHCTF1,1,27447,unified_098332,Human,AHCTF1,ZMYND11
3,chr1:12229781-12230281,VPS13D,AHCTF1,1,17650,unified_008400,Human,AHCTF1,VPS13D
4,chr1:85502682-85503182,ZNHIT6,AHCTF1,1,33475,unified_042415,Human,AHCTF1,ZNHIT6


In [28]:
da_frames = {}

da_summary_rows = []

da_volcano_dir = FIG_DIR / 'volcano_da'


for cell_type in CELL_TYPES:
    plot_df = atac_results[cell_type].merge(annotation_df[['peak_id', 'nearest_gene']], on='peak_id', how='left')
    plot_df['nearest_gene'] = plot_df['nearest_gene'].fillna(plot_df['peak_id'])
    plot_df = prepare_volcano_frame(plot_df, 'nearest_gene')
    da_frames[cell_type] = plot_df

    ultra_robust_peak_ids = load_atac_ultra_robust_peak_ids(cell_type)
    out_base = da_volcano_dir / f'{output_slug(cell_type)}__da_volcano_human_vs_rest'
    da_summary_rows.append(
        plot_volcano_for_cell_type(
            plot_df,
            cell_type,
            out_base,
            ultra_robust_peak_ids=ultra_robust_peak_ids,
        )
    )


da_summary = pd.DataFrame(da_summary_rows)

display(da_summary)

print('Saved DA volcano plots under:', da_volcano_dir)

,cell_type,n_total,n_human_high,n_rest_high,pdf_path,png_path,n_ultra_robust,pdf_path_ultra_robust,png_path_ultra_robust
0,Enterocytes,231226,26655,3999,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,6232,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
1,Macrophages,131521,18194,3515,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,1009,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
2,Naive B cells,67101,4286,605,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,548,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
3,Plasma B cells,137407,18693,4126,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,1292,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
4,T cells,226060,19340,4140,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,932,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...


Saved DA volcano plots under: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/figures/volcano_da


In [32]:
da_provenance_rows = []

for cell_type in CELL_TYPES:
    source_path = ATAC_RESULTS_DIR / celltype_to_dirname(cell_type) / f'{CONTRAST}.csv'
    raw_df = pd.read_csv(source_path)
    filtered_df = raw_df.dropna(subset=['peak_id', 'log2FoldChange', 'padj']).copy()
    filtered_df['neg_log10_padj'] = -np.log10(filtered_df['padj'].clip(lower=1e-300))
    top_peak = filtered_df.sort_values(['padj', 'baseMean'], ascending=[True, False]).iloc[0]['peak_id'] if not filtered_df.empty else None

    da_provenance_rows.append({
        'cell_type': cell_type,
        'source_path': str(source_path),
        'n_total_rows': len(raw_df),
        'n_plotted_rows': len(filtered_df),
        'n_sig_human_high': int(((filtered_df['padj'] < PADJ_THRESH) & (filtered_df['log2FoldChange'] >= LFC_THRESH)).sum()),
        'n_sig_rest_high': int(((filtered_df['padj'] < PADJ_THRESH) & (filtered_df['log2FoldChange'] <= -LFC_THRESH)).sum()),
        'lfc_abs_q995': float(filtered_df['log2FoldChange'].abs().quantile(0.995)) if not filtered_df.empty else np.nan,
        'neg_log10_q995': float(filtered_df['neg_log10_padj'].quantile(0.995)) if not filtered_df.empty else np.nan,
        'top_peak': top_peak,
    })


da_provenance_summary_df = pd.DataFrame(da_provenance_rows)

da_similarity_pairs = [
    ('Enterocytes', 'Plasma B cells'),
    ('Enterocytes', 'T cells'),
    ('Plasma B cells', 'T cells'),
]

da_similarity_rows = []

for cell_type_a, cell_type_b in da_similarity_pairs:
    a_df = atac_results[cell_type_a][['peak_id', 'log2FoldChange', 'padj']].dropna(subset=['peak_id', 'log2FoldChange', 'padj']).copy()
    b_df = atac_results[cell_type_b][['peak_id', 'log2FoldChange', 'padj']].dropna(subset=['peak_id', 'log2FoldChange', 'padj']).copy()
    merged = a_df.merge(b_df, on='peak_id', suffixes=('_a', '_b'))

    sig_a = set(a_df.loc[(a_df['padj'] < PADJ_THRESH) & (a_df['log2FoldChange'].abs() >= LFC_THRESH), 'peak_id'])
    sig_b = set(b_df.loc[(b_df['padj'] < PADJ_THRESH) & (b_df['log2FoldChange'].abs() >= LFC_THRESH), 'peak_id'])
    union_size = len(sig_a | sig_b)

    da_similarity_rows.append({
        'cell_type_a': cell_type_a,
        'cell_type_b': cell_type_b,
        'shared_peak_rows': len(merged),
        'lfc_correlation': float(merged['log2FoldChange_a'].corr(merged['log2FoldChange_b'])) if not merged.empty else np.nan,
        'neg_log10_padj_correlation': float(((-np.log10(merged['padj_a'].clip(lower=1e-300))).corr(-np.log10(merged['padj_b'].clip(lower=1e-300))))) if not merged.empty else np.nan,
        'significant_peak_jaccard': float(len(sig_a & sig_b) / union_size) if union_size else np.nan,
    })


da_similarity_df = pd.DataFrame(da_similarity_rows)

da_provenance_summary_df.to_csv(TABLE_DIR / 'da_provenance_summary.tsv', sep='\t', index=False)

da_similarity_df.to_csv(TABLE_DIR / 'da_similarity_summary.tsv', sep='\t', index=False)

display(da_provenance_summary_df)
display(da_similarity_df)
print('Saved DA provenance summaries to:', TABLE_DIR)

,cell_type,source_path,n_total_rows,n_plotted_rows,n_sig_human_high,n_sig_rest_high,lfc_abs_q995,neg_log10_q995,top_peak
0,Enterocytes,/home/jjanssens/jjans/analysis/adult_intestine...,366077,231226,26655,3999,4.443416,8.381322,unified_340193
1,Macrophages,/home/jjanssens/jjans/analysis/adult_intestine...,131522,131521,18194,3515,3.765923,8.171514,unified_001089
2,Naive B cells,/home/jjanssens/jjans/analysis/adult_intestine...,89714,67101,4286,605,3.936325,6.270344,unified_1015075
3,Plasma B cells,/home/jjanssens/jjans/analysis/adult_intestine...,137409,137407,18693,4126,3.879464,10.620377,unified_340193
4,T cells,/home/jjanssens/jjans/analysis/adult_intestine...,230533,226060,19340,4140,3.554447,9.849878,unified_905964


,cell_type_a,cell_type_b,shared_peak_rows,lfc_correlation,neg_log10_padj_correlation,significant_peak_jaccard
0,Enterocytes,Plasma B cells,86743,0.495084,0.139590,0.079064
1,Enterocytes,T cells,129982,0.469639,0.100953,0.090312
2,Plasma B cells,T cells,107231,0.629019,0.431157,0.197254


Saved DA provenance summaries to: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/tables


In [34]:
de_frames = {}

de_summary_rows = []

de_volcano_dir = FIG_DIR / 'volcano_de'


for cell_type in CELL_TYPES:

    plot_df = prepare_volcano_frame(rna_results[cell_type].copy(), 'gene')

    de_frames[cell_type] = plot_df

    out_base = de_volcano_dir / f'{output_slug(cell_type)}__de_volcano_human_vs_rest'

    de_summary_rows.append(
        plot_volcano_for_cell_type(
            plot_df,
            cell_type,
            out_base,
            n_labels=5,
            include_labels_in_main=True,
            save_labeled_variant=True,
            save_no_label_variant=True,
        )
    )


de_summary = pd.DataFrame(de_summary_rows)

display(de_summary)

print('Saved DE volcano plots under:', de_volcano_dir)

,cell_type,n_total,n_human_high,n_rest_high,n_requested_labels,pdf_path,png_path,pdf_path_labeled,png_path_labeled,pdf_path_no_labels,png_path_no_labels,n_ultra_robust,pdf_path_ultra_robust,png_path_ultra_robust
0,Enterocytes,10905,1132,1334,10,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,0,None,None
1,Macrophages,11162,729,620,10,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,0,None,None
2,Naive B cells,9813,439,393,10,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,0,None,None
3,Plasma B cells,11103,1134,734,10,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,0,None,None
4,T cells,11482,1227,673,10,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,0,None,None


Saved DE volcano plots under: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/figures/volcano_de


In [8]:
def prepare_gene_scores(rna_df: pd.DataFrame, prefix: str) -> pd.DataFrame:

    out = rna_df[['gene', 'gene_key', 'baseMean', 'log2FoldChange', 'padj']].dropna(subset=['gene_key', 'log2FoldChange', 'padj']).copy()

    out = out.sort_values(['padj', 'baseMean'], ascending=[True, False]).drop_duplicates(subset=['gene_key'])

    return out.rename(columns={

        'gene': f'{prefix}_gene',

        'baseMean': f'{prefix}_baseMean',

        'log2FoldChange': f'{prefix}_lfc',

        'padj': f'{prefix}_padj',

    })





def rank_triplets_for_cell_type(cell_type: str) -> pd.DataFrame:

    triplets = scenic_triplets.copy()

    da_df = atac_results[cell_type][['peak_id', 'baseMean', 'log2FoldChange', 'padj']].dropna(subset=['peak_id', 'log2FoldChange', 'padj']).copy()

    da_df = da_df.rename(columns={'baseMean': 'da_baseMean', 'log2FoldChange': 'da_lfc', 'padj': 'da_padj'})



    tf_scores = prepare_gene_scores(rna_results[cell_type], 'tf')

    target_scores = prepare_gene_scores(rna_results[cell_type], 'target')



    scored = triplets.merge(da_df, on='peak_id', how='inner')

    scored = scored.merge(tf_scores[['gene_key', 'tf_gene', 'tf_baseMean', 'tf_lfc', 'tf_padj']], left_on='TF_key', right_on='gene_key', how='inner')

    scored = scored.drop(columns=['gene_key'])

    scored = scored.merge(target_scores[['gene_key', 'target_gene', 'target_baseMean', 'target_lfc', 'target_padj']], left_on='Gene_key', right_on='gene_key', how='inner')

    scored = scored.drop(columns=['gene_key'])



    scored['da_score'] = component_score(scored['da_lfc'], scored['da_padj'])

    scored['tf_score'] = component_score(scored['tf_lfc'], scored['tf_padj'])

    scored['target_score'] = component_score(scored['target_lfc'], scored['target_padj'])



    scored['da_sign'] = np.sign(scored['da_lfc']).astype(int)

    scored['tf_sign'] = np.sign(scored['tf_lfc']).astype(int)

    scored['target_sign'] = np.sign(scored['target_lfc']).astype(int)

    scored = scored[(scored['da_sign'] != 0) & (scored['tf_sign'] != 0) & (scored['target_sign'] != 0)].copy()

    scored = scored[(scored['da_sign'] == scored['tf_sign']) & (scored['da_sign'] == scored['target_sign'])].copy()



    scored['da_sig'] = (scored['da_padj'] < PADJ_THRESH) & (scored['da_lfc'].abs() >= LFC_THRESH)

    scored['tf_sig'] = (scored['tf_padj'] < PADJ_THRESH) & (scored['tf_lfc'].abs() >= LFC_THRESH)

    scored['target_sig'] = (scored['target_padj'] < PADJ_THRESH) & (scored['target_lfc'].abs() >= LFC_THRESH)

    scored['core_sig'] = scored['da_sig'] & scored['target_sig']

    scored['triplet_sig'] = scored['core_sig'] & scored['tf_sig']

    scored['sig_metric'] = significance_metric(scored['da_padj'], scored['target_padj'])

    scored['triplet_sig_metric'] = significance_metric(scored['da_padj'], scored['target_padj'], scored['tf_padj'])



    scored['direction'] = np.where(scored['da_sign'] > 0, 'Human up', 'Rest up')

    scored['total_score'] = scored['da_score'] + scored['tf_score'] + scored['target_score']

    scored['total_score_abs'] = scored['total_score'].abs()

    scored['cell_type'] = cell_type



    scored = scored.sort_values(

        ['direction', 'core_sig', 'triplet_sig', 'total_score_abs', 'sig_metric', 'triplet_rank'],

        ascending=[True, False, False, False, False, True],

    )

    return scored





ranked_triplets = {cell_type: rank_triplets_for_cell_type(cell_type) for cell_type in CELL_TYPES}

ranked_triplet_df = pd.concat(ranked_triplets.values(), ignore_index=True)



ranking_summary = []

for cell_type in CELL_TYPES:

    df = ranked_triplets[cell_type]

    ranking_summary.append({

        'cell_type': cell_type,

        'n_triplets': len(df),

        'n_core_sig': int(df['core_sig'].sum()),

        'n_triplet_sig': int(df['triplet_sig'].sum()),

        'n_human_up_core_sig': int(((df['direction'] == 'Human up') & df['core_sig']).sum()),

        'top_human_up_score': float(df.loc[df['direction'] == 'Human up', 'total_score'].max()) if (df['direction'] == 'Human up').any() else np.nan,

    })



ranking_summary_df = pd.DataFrame(ranking_summary)

display(ranking_summary_df)

display(ranked_triplets['Enterocytes'][['TF', 'Gene', 'peak_id', 'direction', 'da_sig', 'target_sig', 'tf_sig', 'total_score']].head(10))

,cell_type,n_triplets,n_core_sig,n_triplet_sig,n_human_up_core_sig,top_human_up_score
0,Enterocytes,13776,808,134,663,1816.663402
1,Macrophages,7475,196,26,150,1395.928112
2,Naive B cells,4566,84,8,76,452.675218
3,Plasma B cells,7495,420,125,302,1071.681981
4,T cells,9545,270,85,236,1493.618234


,TF,Gene,peak_id,direction,da_sig,target_sig,tf_sig,total_score
37849,CREB3L3,LCT,unified_542631,Human up,True,True,True,78.978690
35576,KLF4,GPR153,unified_003719,Human up,True,True,True,68.036748
35572,KLF4,GPR153,unified_003627,Human up,True,True,True,61.675703
11316,KLF4,ABLIM1,unified_142919,Human up,True,True,True,58.943026
35575,KLF4,GPR153,unified_003717,Human up,True,True,True,57.108152
40747,CREB3L3,C11orf86,unified_177236,Human up,True,True,True,55.298680
35573,KLF4,GPR153,unified_003672,Human up,True,True,True,54.883827
35878,KLF4,POLD4,unified_177572,Human up,True,True,True,49.957960
35883,KLF4,POLD4,unified_177445,Human up,True,True,True,49.714816
40743,CREB3L3,FABP2,unified_742907,Human up,True,True,True,47.984970


In [9]:
ranked_triplet_df.to_csv(TABLE_DIR / 'all_ranked_triplets.tsv', sep='\t', index=False)

ranking_summary_df.to_csv(TABLE_DIR / 'triplet_ranking_summary.tsv', sep='\t', index=False)



highlight_triplets_per_cell = {}

highlight_summary_rows = []



for cell_type in CELL_TYPES:

    df = ranked_triplets[cell_type]

    primary = df[(df['direction'] == 'Human up') & df['core_sig']].copy()

    selected = select_nonredundant_triplets(primary, annotation_df, n=TOP_TRIPLETS_TO_PLOT)



    if len(selected) < TOP_TRIPLETS_TO_PLOT:

        fallback = df[(df['direction'] == 'Human up') & (~df.index.isin(selected.index))].copy()

        fallback = select_nonredundant_triplets(fallback, annotation_df, n=TOP_TRIPLETS_TO_PLOT - len(selected))

        selected = pd.concat([selected, fallback], ignore_index=False)



    selected = selected.head(TOP_TRIPLETS_TO_PLOT).copy()

    selected['triplet_label'] = selected['TF'].astype(str) + ' -> ' + selected['Gene'].astype(str)

    highlight_triplets_per_cell[cell_type] = selected



    safe_name = output_slug(cell_type)

    df.to_csv(TABLE_DIR / f'{safe_name}__triplet_ranking.tsv', sep='\t', index=False)

    selected.to_csv(TABLE_DIR / f'{safe_name}__selected_top{TOP_TRIPLETS_TO_PLOT}_triplets.tsv', sep='\t', index=False)



    highlight_summary_rows.append({

        'cell_type': cell_type,

        'n_selected': len(selected),

        'n_selected_core_sig': int(selected['core_sig'].sum()),

        'n_selected_triplet_sig': int(selected['triplet_sig'].sum()),

        'top_triplet': None if selected.empty else selected.iloc[0]['triplet_label'],

    })



top_triplets_per_cell = highlight_triplets_per_cell

highlight_summary_df = pd.DataFrame(highlight_summary_rows)

display(highlight_summary_df)

print('Saved ranked triplet tables to:', TABLE_DIR)

for cell_type in CELL_TYPES:

    print(cell_type)

    display(highlight_triplets_per_cell[cell_type][['TF', 'Gene', 'peak_id', 'da_sig', 'target_sig', 'tf_sig', 'total_score']].head(10))

,cell_type,n_selected,n_selected_core_sig,n_selected_triplet_sig,top_triplet
0,Enterocytes,10,10,10,CREB3L3 -> LCT
1,Macrophages,10,10,10,NFIL3 -> THBS1
2,Naive B cells,10,10,7,PPARA -> UGT8
3,Plasma B cells,10,10,10,KLF4 -> FXYD3
4,T cells,10,10,10,EHF -> FXYD3


Saved ranked triplet tables to: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/tables
Enterocytes


,TF,Gene,peak_id,da_sig,target_sig,tf_sig,total_score
37849,CREB3L3,LCT,unified_542631,True,True,True,78.978690
35576,KLF4,GPR153,unified_003719,True,True,True,68.036748
35572,KLF4,GPR153,unified_003627,True,True,True,61.675703
11316,KLF4,ABLIM1,unified_142919,True,True,True,58.943026
40747,CREB3L3,C11orf86,unified_177236,True,True,True,55.298680
35573,KLF4,GPR153,unified_003672,True,True,True,54.883827
35878,KLF4,POLD4,unified_177572,True,True,True,49.957960
35883,KLF4,POLD4,unified_177445,True,True,True,49.714816
40743,CREB3L3,FABP2,unified_742907,True,True,True,47.984970
9289,CREB3L3,WWC1,unified_805955,True,True,True,46.744209


Macrophages


,TF,Gene,peak_id,da_sig,target_sig,tf_sig,total_score
26451,NFIL3,THBS1,unified_318089,True,True,True,93.010463
25846,SOX5,ADAMTS2,unified_810968,True,True,True,50.118640
25731,SOX5,NNMT,unified_193718,True,True,True,32.725359
23920,FOSL1,FOSL1,unified_176313,True,True,True,32.400240
25730,SOX5,NNMT,unified_193742,True,True,True,30.562252
23919,FOSL1,FOSL1,unified_176292,True,True,True,29.157644
25761,SOX5,SSPN,unified_215354,True,True,True,24.732005
23464,PLAGL1,EMP2,unified_355062,True,True,True,22.825453
11387,PLAGL1,FAR2,unified_216534,True,True,True,20.266664
27143,ATOH1,FFAR4,unified_132798,True,True,True,18.841100


...

,TF,Gene,peak_id,da_sig,target_sig,tf_sig,total_score
12727,PPARA,UGT8,unified_742128,True,True,True,87.536219
13068,PPARA,CDR2,unified_361205,True,True,True,45.296549
1382,KLF4,CLTB,unified_809396,True,True,True,25.415910
13214,PPARA,MT1X,unified_369874,True,True,True,16.081731
14389,PPARA,NIPAL2,unified_952110,True,True,True,14.537848
15665,PPARA,TP53I3,unified_503460,True,True,True,13.669051
13179,PPARA,HSD11B2,unified_373240,True,True,True,9.482949
9259,MEF2C,RIPOR1,unified_373364,True,True,False,38.238055
6376,FLI1,ARL4C,unified_573681,True,True,False,37.382507
14902,EHF,ST6GALNAC1,unified_430154,True,True,False,28.851974


Plasma B cells


,TF,Gene,peak_id,da_sig,target_sig,tf_sig,total_score
22414,KLF4,FXYD3,unified_479403,True,True,True,293.722609
55,ATF3,AREG,unified_731251,True,True,True,62.958458
22362,KLF4,GIPR,unified_486390,True,True,True,57.295137
18137,EGR1,FOSB,unified_486058,True,True,True,48.777586
18882,PPARA,HDHD3,unified_998931,True,True,True,46.514246
14714,KLF4,TMEM54,unified_023007,True,True,True,44.539543
18278,CREB3L3,SSUH2,unified_646866,True,True,True,43.023605
17279,ATOH1,CDC42EP3,unified_509178,True,True,True,42.471541
18201,CREB3L3,FABP2,unified_742907,True,True,True,42.423712
8261,PLAGL1,CAMK1D,unified_104403,True,True,True,40.247867


T cells


,TF,Gene,peak_id,da_sig,target_sig,tf_sig,total_score
28274,EHF,FXYD3,unified_479253,True,True,True,524.554037
477,KLF4,CEACAM7,unified_483641,True,True,True,121.022433
31893,CREB3L3,CIDEB,unified_282478,True,True,True,99.656138
28233,EHF,SLC26A2,unified_800263,True,True,True,94.426243
28387,EHF,SLC46A1,unified_402242,True,True,True,90.015319
34501,KLF4,GIPR,unified_486390,True,True,True,84.808865
26632,EHF,ST6GALNAC1,unified_430154,True,True,True,61.070374
28521,EHF,PRAP1,unified_150949,True,True,True,58.634827
29476,KLF4,B3GNT6,unified_184486,True,True,True,56.757123
14875,EHF,FBXO34,unified_289699,True,True,True,55.714922


In [38]:
triplet_scatter_dir = FIG_DIR / 'triplet_scatter'

scatter_summary_rows = []

labeled_triplets_per_cell = {}


for cell_type in CELL_TYPES:
    all_df = ranked_triplets[cell_type].copy()
    selected_df = all_df[all_df['core_sig']].copy()
    label_df = prepare_triplet_label_df(highlight_triplets_per_cell[cell_type], n_labels=min(6, len(highlight_triplets_per_cell[cell_type])))
    focus_df = select_focus_triplet_for_target(all_df, SPECIAL_TRIPLET_TARGETS.get(cell_type))
    out_base = triplet_scatter_dir / f'{output_slug(cell_type)}__triplet_scatter'

    scatter_info = plot_triplet_scatter_for_cell_type(
        all_df=all_df,
        selected_df=selected_df,
        label_df=label_df,
        out_base=out_base,
        focus_df=focus_df,
    )
    labeled_triplets_per_cell[cell_type] = scatter_info['placed_label_df'].copy()
    scatter_summary_rows.append({
        'cell_type': cell_type,
        'n_total': len(all_df),
        'n_core_sig_plotted': len(selected_df),
        'n_requested_labels': scatter_info['n_requested_labels'],
        'n_placed_labels': scatter_info['n_placed_labels'],
        'focus_target_gene': SPECIAL_TRIPLET_TARGETS.get(cell_type),
        'focus_triplet': None if focus_df.empty else focus_df.iloc[0]['triplet_label'],
        'pdf_path': scatter_info['pdf_path'],
        'png_path': scatter_info['png_path'],
        'pdf_path_no_labels': scatter_info['pdf_path_no_labels'],
        'png_path_no_labels': scatter_info['png_path_no_labels'],
        'pdf_path_focus_only': scatter_info['pdf_path_focus_only'],
        'png_path_focus_only': scatter_info['png_path_focus_only'],
    })


scatter_summary_df = pd.DataFrame(scatter_summary_rows)

display(scatter_summary_df)

print('Saved triplet scatter plots under:', triplet_scatter_dir)

,cell_type,n_total,n_core_sig_plotted,n_requested_labels,n_placed_labels,focus_target_gene,focus_triplet,pdf_path,png_path,pdf_path_no_labels,png_path_no_labels,pdf_path_focus_only,png_path_focus_only
0,Enterocytes,13776,808,6,7,ALPI,CREB3L3 -> ALPI,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
1,Macrophages,7475,196,6,6,None,None,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,None,None
2,Naive B cells,4566,84,6,6,None,None,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,None,None
3,Plasma B cells,7495,420,6,6,AREG,ATF3 -> AREG,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
4,T cells,9545,270,6,6,None,None,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,None,None


Saved triplet scatter plots under: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/figures/triplet_scatter


In [15]:
def bigwig_bundle_for_cell_type(cell_type: str) -> tuple[dict[str, str] | None, str | None]:

    folder = match_bigwig_cell_type(cell_type, available_bigwigs)

    if folder is None:

        return None, None

    return available_bigwigs[folder], folder





track_records = []

selected_expression_records = []

slide_panel_records = []

slide_panel_candidate_summary = []

slide_panel_dir = OUT_DIR / 'triplet_slide_panels'

slide_candidate_dir = TABLE_DIR / 'slide_panel_candidates'

slide_candidate_dir.mkdir(parents=True, exist_ok=True)



for cell_type in CELL_TYPES:

    bw_paths, matched_folder = bigwig_bundle_for_cell_type(cell_type)

    selected_df = highlight_triplets_per_cell[cell_type]



    if selected_df.empty:

        print(f'Skipping {cell_type}: no selected triplets available.')

        continue



    expr_genes = list(dict.fromkeys(selected_df['Gene'].astype(str).tolist() + selected_df['TF'].astype(str).tolist()))

    expr_df = build_species_expression_matrix(cell_type, expr_genes, rna_meta, species_count_cache)

    expr_dir = GENE_SET_HEATMAP_DIR / output_slug(cell_type)

    expr_dir.mkdir(parents=True, exist_ok=True)

    expr_base = expr_dir / 'selected_triplet_genes_expression'

    expr_df.to_csv(expr_base.with_suffix('.tsv'), sep='\t')

    expr_pdf, expr_png, expr_z = plot_species_expression_heatmap(expr_df, expr_base)

    expr_z.to_csv(expr_dir / 'selected_triplet_genes_expression_zscore.tsv', sep='\t')

    selected_expression_records.append({

        'cell_type': cell_type,

        'n_genes': expr_df.shape[0],

        'pdf_path': str(expr_pdf),

        'png_path': str(expr_png),

    })



    panel_candidates = build_panel_candidate_df(ranked_triplets[cell_type], gene_sets)

    panel_candidates_path = slide_candidate_dir / f'{output_slug(cell_type)}__all_de_or_da_significant_triplets.tsv'

    panel_candidates.to_csv(panel_candidates_path, sep='\t', index=False)

    panel_render_df = choose_panel_render_subset(panel_candidates)

    slide_panel_candidate_summary.append({

        'cell_type': cell_type,

        'n_panel_candidates': len(panel_candidates),

        'n_rendered_panels': len(panel_render_df),

        'candidate_table': str(panel_candidates_path),

    })



    if bw_paths is None:

        print(f'Skipping accessibility tracks for {cell_type}: no six-species BigWig folder found.')

        continue



    cell_out_dir = TRACK_DIR / output_slug(cell_type)

    cell_out_dir.mkdir(parents=True, exist_ok=True)

    cell_slide_dir = slide_panel_dir / output_slug(cell_type)

    cell_slide_dir.mkdir(parents=True, exist_ok=True)



    for rank_idx, (_, row) in enumerate(selected_df.iterrows(), start=1):

        region = peak_to_region(annotation_df, row['peak_id'])

        if region is None:

            print(f'Skipping {cell_type} / {row["peak_id"]}: no coordinates available.')

            continue



        stem = f'{rank_idx:02d}_{sanitize_token(row["TF"])}__{sanitize_token(row["Gene"])}__{sanitize_token(row["peak_id"])}'

        out_base = cell_out_dir / stem

        out_pdf, out_png = plot_normalized_accessibility_tracks(bw_paths, region, out_base)

        track_records.append({

            'cell_type': cell_type,

            'bigwig_folder': matched_folder,

            'rank': rank_idx,

            'TF': row['TF'],

            'Gene': row['Gene'],

            'peak_id': row['peak_id'],

            'core_sig': bool(row['core_sig']),

            'triplet_sig': bool(row['triplet_sig']),

            'region': region,

            'output_pdf': str(out_pdf),

            'output_png': str(out_png),

        })



    for panel_idx, (_, row) in enumerate(panel_render_df.iterrows(), start=1):

        region = peak_to_region(annotation_df, row['peak_id'])

        if region is None:

            continue

        stem = f'{panel_idx:03d}_{sanitize_token(row["primary_signature"])}__{sanitize_token(row["TF"])}__{sanitize_token(row["Gene"])}__{sanitize_token(row["peak_id"])}'

        panel_base = cell_slide_dir / stem

        panel_pdf, panel_png, dotplot_df = plot_triplet_slide_panel(

            bw_paths=bw_paths,

            region=region,

            cell_type=cell_type,

            tf_name=str(row['TF']),

            gene_name=str(row['Gene']),

            out_base=panel_base,

        )

        dotplot_path = panel_base.with_suffix('.tsv')

        dotplot_df.to_csv(dotplot_path, sep='\t', index=False)

        slide_panel_records.append({

            'cell_type': cell_type,

            'panel_rank': int(row['panel_rank']),

            'primary_signature': row['primary_signature'],

            'signature_source': row['signature_source'],

            'panel_sig_count': int(row['panel_sig_count']),

            'panel_sig_score': float(row['panel_sig_score']),

            'TF': row['TF'],

            'Gene': row['Gene'],

            'peak_id': row['peak_id'],

            'triplet_label': f"{row['TF']} -> {row['Gene']}",

            'region': region,

            'panel_pdf': str(panel_pdf),

            'panel_png': str(panel_png),

            'dotplot_tsv': str(dotplot_path),

        })



selected_expression_manifest_df = pd.DataFrame(selected_expression_records)

selected_expression_manifest_path = TABLE_DIR / 'selected_triplet_expression_manifest.tsv'

selected_expression_manifest_df.to_csv(selected_expression_manifest_path, sep='\t', index=False)



track_manifest_df = pd.DataFrame(track_records)

track_manifest_path = TABLE_DIR / 'triplet_track_manifest.tsv'

track_manifest_df.to_csv(track_manifest_path, sep='\t', index=False)



slide_panel_manifest_df = pd.DataFrame(slide_panel_records)

slide_panel_manifest_path = TABLE_DIR / 'triplet_slide_panel_manifest.tsv'

slide_panel_manifest_df.to_csv(slide_panel_manifest_path, sep='\t', index=False)



slide_panel_candidate_summary_df = pd.DataFrame(slide_panel_candidate_summary)

slide_panel_candidate_summary_path = TABLE_DIR / 'triplet_slide_panel_candidate_summary.tsv'

slide_panel_candidate_summary_df.to_csv(slide_panel_candidate_summary_path, sep='\t', index=False)



display(selected_expression_manifest_df)

display(track_manifest_df.head(15))

display(slide_panel_candidate_summary_df)

display(slide_panel_manifest_df.head(15))

print('Saved selected-triplet expression manifest:', selected_expression_manifest_path)

print('Saved track manifest:', track_manifest_path)

print('Saved slide-panel candidate summary:', slide_panel_candidate_summary_path)

print('Saved slide-panel manifest:', slide_panel_manifest_path)

print('All ordered panel candidates saved under:', slide_candidate_dir)

print('Track plots saved under:', TRACK_DIR)

print('Slide-ready triplet panels saved under:', slide_panel_dir)

,cell_type,n_genes,pdf_path,png_path
0,Enterocytes,9,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
1,Macrophages,12,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
2,Naive B cells,15,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
3,Plasma B cells,17,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
4,T cells,13,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...


,cell_type,bigwig_folder,rank,TF,Gene,peak_id,core_sig,triplet_sig,region,output_pdf,output_png
0,Enterocytes,enterocytes,1,CREB3L3,LCT,unified_542631,True,True,chr2:135841439-135847939,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
1,Enterocytes,enterocytes,2,KLF4,GPR153,unified_003719,True,True,chr1:6278714-6285214,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
2,Enterocytes,enterocytes,3,KLF4,GPR153,unified_003627,True,True,chr1:6202795-6209295,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
3,Enterocytes,enterocytes,4,KLF4,ABLIM1,unified_142919,True,True,chr10:114496696-114503196,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
4,Enterocytes,enterocytes,5,CREB3L3,C11orf86,unified_177236,True,True,chr11:66947064-66953564,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
5,Enterocytes,enterocytes,6,KLF4,GPR153,unified_003672,True,True,chr1:6243631-6250131,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
6,Enterocytes,enterocytes,7,KLF4,POLD4,unified_177572,True,True,chr11:67342899-67349399,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
7,Enterocytes,enterocytes,8,KLF4,POLD4,unified_177445,True,True,chr11:67236695-67243195,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
8,Enterocytes,enterocytes,9,CREB3L3,FABP2,unified_742907,True,True,chr4:119451037-119457537,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
9,Enterocytes,enterocytes,10,CREB3L3,WWC1,unified_805955,True,True,chr5:168314629-168321129,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...


,cell_type,n_panel_candidates,n_rendered_panels,candidate_table
0,Enterocytes,7483,58,/home/jjanssens/jjans/analysis/adult_intestine...
1,Macrophages,2772,57,/home/jjanssens/jjans/analysis/adult_intestine...
2,Naive B cells,1138,47,/home/jjanssens/jjans/analysis/adult_intestine...
3,Plasma B cells,3747,55,/home/jjanssens/jjans/analysis/adult_intestine...
4,T cells,3330,60,/home/jjanssens/jjans/analysis/adult_intestine...


,cell_type,panel_rank,primary_signature,signature_source,panel_sig_count,panel_sig_score,TF,Gene,peak_id,triplet_label,region,panel_pdf,panel_png,dotplot_tsv
0,Enterocytes,1,bmp_tgfb,target,2,3.998639,EHF,PITX2,unified_741145,EHF -> PITX2,chr4:110616487-110622987,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
1,Enterocytes,2,bmp_tgfb,target,2,3.998639,ETV6,PITX2,unified_741145,ETV6 -> PITX2,chr4:110616487-110622987,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
2,Enterocytes,3,bmp_tgfb,target,2,7.527462,TRPS1,SKI,unified_001740,TRPS1 -> SKI,chr1:2297957-2304457,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
3,Enterocytes,91,cytokines_chemokines,target,3,10.370662,CREB3L3,TNFRSF1A,unified_207411,CREB3L3 -> TNFRSF1A,chr12:6206988-6213488,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
4,Enterocytes,92,cytokines_chemokines,target,3,7.623089,CREB3L3,TNFRSF1A,unified_207537,CREB3L3 -> TNFRSF1A,chr12:6307386-6313886,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
5,Enterocytes,93,cytokines_chemokines,target,2,13.042792,KLF3,WNT2B,unified_051120,KLF3 -> WNT2B,chr1:112391069-112397569,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
6,Enterocytes,192,ecm_adhesion,target,3,21.486590,KLF12,UTRN,unified_861137,KLF12 -> UTRN,chr6:144283940-144290440,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
7,Enterocytes,193,ecm_adhesion,target,3,21.139996,FLI1,UTRN,unified_861190,FLI1 -> UTRN,chr6:144340667-144347167,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
8,Enterocytes,194,ecm_adhesion,target,3,20.734987,ERG,UTRN,unified_861190,ERG -> UTRN,chr6:144340667-144347167,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
9,Enterocytes,500,enteroendocrine,target,2,3.163017,KLF4,SYT7,unified_172147,KLF4 -> SYT7,chr11:61531477-61537977,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...


Saved selected-triplet expression manifest: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/tables/selected_triplet_expression_manifest.tsv
Saved track manifest: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/tables/triplet_track_manifest.tsv
Saved slide-panel candidate summary: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/tables/triplet_slide_panel_candidate_summary.tsv
Saved slide-panel manifest: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/tables/triplet_slide_panel_manifest.tsv
All ordered panel candidates saved under: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/tables/slide_panel_candidates
Track plots saved under: /home/jjanssens/jjans/analysis/adult

In [22]:
gene_set_table_dir = TABLE_DIR / 'gene_sets'

gene_set_table_dir.mkdir(parents=True, exist_ok=True)

gene_set_summary_rows = []



for cell_type in CELL_TYPES:

    all_df = ranked_triplets[cell_type].copy()

    sig_df = all_df[all_df['core_sig']].copy()

    if sig_df.empty:

        continue



    cell_scatter_dir = GENE_SET_FIG_DIR / output_slug(cell_type)

    cell_heatmap_dir = GENE_SET_HEATMAP_DIR / 'gene_sets' / output_slug(cell_type)

    cell_scatter_dir.mkdir(parents=True, exist_ok=True)

    cell_heatmap_dir.mkdir(parents=True, exist_ok=True)



    for gene_set_name, gene_set_genes in sorted(gene_sets.items()):

        matched = sig_df[sig_df['Gene_key'].isin(gene_set_genes)].copy()

        if matched.empty:

            continue



        set_slug = output_slug(gene_set_name)

        matched = matched.sort_values(['total_score_abs', 'sig_metric', 'triplet_rank'], ascending=[False, False, True]).copy()

        matched.to_csv(gene_set_table_dir / f'{output_slug(cell_type)}__{set_slug}__triplets.tsv', sep='\t', index=False)



        label_candidates = matched[matched['direction'] == 'Human up'].copy()

        if label_candidates.empty:

            label_candidates = matched.copy()

        label_candidates = select_nonredundant_triplets(label_candidates, annotation_df, n=min(TOP_LABELS, len(label_candidates)))

        label_df = prepare_triplet_label_df(label_candidates, n_labels=min(TOP_LABELS, len(label_candidates)))



        scatter_base = cell_scatter_dir / f'{set_slug}__triplet_scatter'

        scatter_info = plot_triplet_scatter_for_cell_type(

            all_df=all_df,

            selected_df=matched,

            label_df=label_df,

            out_base=scatter_base,

        )



        target_gene_order = (

            matched[['Gene', 'Gene_key', 'total_score_abs', 'sig_metric']]

            .sort_values(['total_score_abs', 'sig_metric'], ascending=[False, False])

            .drop_duplicates(subset=['Gene_key'])['Gene']

            .tolist()

        )

        expr_df = build_species_expression_matrix(cell_type, target_gene_order, rna_meta, species_count_cache)

        expr_base = cell_heatmap_dir / f'{set_slug}__target_expression'

        expr_df.to_csv(expr_base.with_suffix('.tsv'), sep='\t')

        expr_pdf, expr_png, expr_z = plot_species_expression_heatmap(expr_df, expr_base)

        expr_z.to_csv(cell_heatmap_dir / f'{set_slug}__target_expression_zscore.tsv', sep='\t')



        gene_set_summary_rows.append({

            'cell_type': cell_type,

            'gene_set': gene_set_name,

            'n_triplets': len(matched),

            'n_unique_targets': len(target_gene_order),

            'scatter_pdf': scatter_info['pdf_path'],

            'scatter_pdf_no_labels': scatter_info['pdf_path_no_labels'],

            'heatmap_pdf': str(expr_pdf),

        })



gene_set_summary_df = pd.DataFrame(gene_set_summary_rows)

gene_set_summary_path = gene_set_table_dir / 'gene_set_triplet_summary.tsv'

gene_set_summary_df.to_csv(gene_set_summary_path, sep='\t', index=False)

display(gene_set_summary_df.head(20))

print('Saved gene-set triplet outputs under:', gene_set_table_dir)

print('Saved gene-set scatter plots under:', GENE_SET_FIG_DIR)

print('Saved gene-set expression heatmaps under:', GENE_SET_HEATMAP_DIR / 'gene_sets')

,cell_type,gene_set,n_triplets,n_unique_targets,scatter_pdf,heatmap_pdf
0,Enterocytes,bmp_tgfb,2,1,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
1,Enterocytes,cytokines_chemokines,38,6,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
2,Enterocytes,ecm_adhesion,59,12,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
3,Enterocytes,epithelial_barrier,19,6,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
4,Enterocytes,epithelial_barrier_manual,6,2,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
5,Enterocytes,growth_factor_receptors,5,3,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
6,Enterocytes,growth_factors,20,2,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
7,Enterocytes,human_tfs,56,18,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
8,Enterocytes,immune_checkpoint,4,4,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
9,Enterocytes,immune_receptors,7,3,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...


Saved gene-set triplet outputs under: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/tables/gene_sets
Saved gene-set scatter plots under: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/gene_set_triplet_figures
Saved gene-set expression heatmaps under: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/gene_set_expression_heatmaps/gene_sets


In [22]:
import gzip
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.axes import Axes
from matplotlib.colors import TwoSlopeNorm
from matplotlib.figure import Figure
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter, MaxNLocator


PEAK_QUERY = 'unified_244384'
PEAK_QUERY_CELL_TYPE = 'Plasma B cells'
PEAK_QUERY_FORCE_GENES = ['OAS1', 'OAS2', 'OAS3']
PEAK_QUERY_PANEL_SPECS = [
    ('oas1', ['OAS1']),
    ('oas1_oas2', ['OAS1', 'OAS2']),
    ('oas_all', ['OAS1', 'OAS2', 'OAS3']),
]
PEAK_QUERY_LOCUS_FLANK = 10000

TRIPLET_TARGET_GENES = ['AREG', 'ALPI']
TRIPLET_LOCUS_FLANK = 10000

peak_query_dir = OUT_DIR / 'peak_queries' / output_slug(PEAK_QUERY_CELL_TYPE)
peak_query_dir.mkdir(parents=True, exist_ok=True)

triplet_locus_dir = OUT_DIR / 'triplet_target_gene_locus_panels'
triplet_locus_dir.mkdir(parents=True, exist_ok=True)


def load_gene_coordinates(gtf_path: Path, gene_names: list[str]) -> pd.DataFrame:
    gene_set = {str(gene) for gene in gene_names}
    rows = []
    with gzip.open(gtf_path, 'rt') as handle:
        for line in handle:
            if line.startswith('#'):
                continue
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 9 or parts[2] != 'gene':
                continue
            match = re.search(r'gene_name "([^"]+)"', parts[8])
            if match is None:
                continue
            gene_name = match.group(1)
            if gene_name not in gene_set:
                continue
            rows.append({
                'gene_name': gene_name,
                'chrom': parts[0],
                'start': int(parts[3]),
                'end': int(parts[4]),
            })
    return pd.DataFrame(rows).drop_duplicates().sort_values(['start', 'end']).reset_index(drop=True)


def load_gene_exons(gtf_path: Path, gene_names: list[str]) -> pd.DataFrame:
    gene_set = {str(gene) for gene in gene_names}
    rows = []
    with gzip.open(gtf_path, 'rt') as handle:
        for line in handle:
            if line.startswith('#'):
                continue
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 9 or parts[2] != 'exon':
                continue
            match = re.search(r'gene_name "([^"]+)"', parts[8])
            if match is None:
                continue
            gene_name = match.group(1)
            if gene_name not in gene_set:
                continue
            rows.append({
                'gene_name': gene_name,
                'chrom': parts[0],
                'start': int(parts[3]),
                'end': int(parts[4]),
            })
    return pd.DataFrame(rows).drop_duplicates().sort_values(['gene_name', 'start', 'end']).reset_index(drop=True)


def load_region_gene_models(gtf_path: Path, chrom: str, start: int, end: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    gene_rows = []
    exon_rows = []
    with gzip.open(gtf_path, 'rt') as handle:
        for line in handle:
            if line.startswith('#'):
                continue
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 9 or parts[2] not in {'gene', 'exon'}:
                continue
            row_chrom = str(parts[0])
            row_start = int(parts[3])
            row_end = int(parts[4])
            if row_chrom != str(chrom) or row_end < start or row_start > end:
                continue
            gene_match = re.search(r'gene_name "([^"]+)"', parts[8])
            if gene_match is None:
                gene_match = re.search(r'gene_id "([^"]+)"', parts[8])
            if gene_match is None:
                continue
            row = {
                'gene_name': gene_match.group(1),
                'chrom': row_chrom,
                'start': row_start,
                'end': row_end,
            }
            if parts[2] == 'gene':
                gene_rows.append(row)
            else:
                exon_rows.append(row)
    gene_df = pd.DataFrame(gene_rows).drop_duplicates().sort_values(['start', 'end']).reset_index(drop=True)
    exon_df = pd.DataFrame(exon_rows).drop_duplicates().sort_values(['gene_name', 'start', 'end']).reset_index(drop=True)
    return gene_df, exon_df


def merge_intervals(intervals: list[tuple[int, int]]) -> list[tuple[int, int]]:
    if not intervals:
        return []
    merged = [list(intervals[0])]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return [(int(start), int(end)) for start, end in merged]


def compute_atac_contrast_score(log2fc: pd.Series, padj: pd.Series) -> pd.Series:
    safe_padj = pd.to_numeric(padj, errors='coerce').clip(lower=1e-300, upper=1.0)
    safe_lfc = pd.to_numeric(log2fc, errors='coerce')
    return -np.log10(safe_padj) * safe_lfc


def build_panel_region(
    all_gene_coords: pd.DataFrame,
    anchor_genes: list[str],
    flank: int = PEAK_QUERY_LOCUS_FLANK,
) -> tuple[str, int, int, pd.DataFrame]:
    anchor_coords = all_gene_coords[all_gene_coords['gene_name'].isin(anchor_genes)].copy()
    if anchor_coords.empty:
        raise ValueError(f'Could not find coordinates for anchor genes: {anchor_genes}')
    chrom = str(anchor_coords['chrom'].iloc[0])
    region_start = max(0, int(anchor_coords['start'].min()) - flank)
    region_end = int(anchor_coords['end'].max()) + flank
    region_gene_coords = all_gene_coords[
        (all_gene_coords['chrom'].astype(str) == chrom)
        & (all_gene_coords['start'] <= region_end)
        & (all_gene_coords['end'] >= region_start)
    ].copy().sort_values(['start', 'end']).reset_index(drop=True)
    region = f'{chrom}:{region_start}-{region_end}'
    return region, region_start, region_end, region_gene_coords


def build_target_peak_region(
    annotation_df: pd.DataFrame,
    target_gene_coords: pd.DataFrame,
    peak_id: str,
    flank: int = TRIPLET_LOCUS_FLANK,
) -> tuple[str, int, int]:
    peak_row = annotation_df.loc[annotation_df['peak_id'].astype(str) == str(peak_id), ['chrom', 'start', 'end']].dropna().head(1)
    if peak_row.empty:
        raise ValueError(f'Could not find coordinates for peak: {peak_id}')
    chrom = str(target_gene_coords['chrom'].iloc[0])
    peak_chrom = str(peak_row['chrom'].iloc[0])
    if peak_chrom != chrom:
        raise ValueError(f'Peak {peak_id} is on {peak_chrom}, target gene is on {chrom}.')
    region_start = max(0, min(int(target_gene_coords['start'].min()), int(peak_row['start'].iloc[0])) - flank)
    region_end = max(int(target_gene_coords['end'].max()), int(peak_row['end'].iloc[0])) + flank
    return f'{chrom}:{region_start}-{region_end}', region_start, region_end


def collect_scored_peaks(
    annotation_df: pd.DataFrame,
    atac_df: pd.DataFrame,
    chrom: str,
    region_start: int,
    region_end: int,
) -> pd.DataFrame:
    cols = ['peak_id', 'chrom', 'start', 'end']
    subset = annotation_df[cols].dropna(subset=['chrom', 'start', 'end']).copy()
    subset = subset[subset['chrom'].astype(str) == str(chrom)].copy()
    subset = subset[(subset['start'] <= region_end) & (subset['end'] >= region_start)].copy()
    if subset.empty:
        return pd.DataFrame(columns=['peak_id', 'chrom', 'start', 'end', 'peak_region', 'atac_log2fc', 'atac_padj', 'contrast_score'])
    atac_scores = atac_df[['peak_id', 'log2FoldChange', 'padj']].drop_duplicates(subset=['peak_id']).copy()
    atac_scores = atac_scores.rename(columns={'log2FoldChange': 'atac_log2fc', 'padj': 'atac_padj'})
    subset = subset.merge(atac_scores, on='peak_id', how='left')
    subset['contrast_score'] = compute_atac_contrast_score(subset['atac_log2fc'], subset['atac_padj'])
    subset['peak_region'] = (
        subset['chrom'].astype(str)
        + ':'
        + subset['start'].astype(int).astype(str)
        + '-'
        + subset['end'].astype(int).astype(str)
    )
    return subset.sort_values(['start', 'end', 'peak_id']).reset_index(drop=True)


def export_peak_table(scored_peak_df: pd.DataFrame, highlight_peak_id: str | None = None) -> pd.DataFrame:
    columns = ['peak_id', 'region', 'contrast_score', 'atac_log2fc', 'atac_padj', 'is_highlight']
    if scored_peak_df.empty:
        return pd.DataFrame(columns=columns)
    out = scored_peak_df[['peak_id', 'peak_region', 'contrast_score', 'atac_log2fc', 'atac_padj']].copy()
    out = out.rename(columns={'peak_region': 'region'})
    out['is_highlight'] = out['peak_id'].astype(str) == str(highlight_peak_id) if highlight_peak_id is not None else False
    return out


def prepare_peak_query_dotplot_display(dotplot_df: pd.DataFrame, gene_order: list[str]) -> tuple[pd.DataFrame, list[str], dict[str, float]]:
    if dotplot_df.empty:
        return dotplot_df.copy(), [], {}
    available_genes = set(dotplot_df['gene_label'].astype(str))
    display_genes = [gene for gene in gene_order if gene in available_genes]
    if not display_genes:
        return dotplot_df.iloc[0:0].copy(), [], {}
    spacing = 0.36
    gene_positions = {gene: idx * spacing for idx, gene in enumerate(display_genes)}
    plot_df = dotplot_df[dotplot_df['gene_label'].isin(display_genes)].copy()
    plot_df['gene_position'] = plot_df['gene_label'].map(gene_positions)
    plot_df['avg_expr_minmax'] = plot_df.groupby('gene_label')['avg_expr'].transform(
        lambda values: pd.Series(0.5, index=values.index)
        if float(values.max() - values.min()) == 0.0
        else (values - values.min()) / (values.max() - values.min())
    )
    plot_df['avg_expr_minmax'] = plot_df['avg_expr_minmax'].clip(0.0, 1.0).fillna(0.5)
    plot_df['size'] = 24 + (plot_df['pct_expr'].clip(lower=0, upper=100) / 100.0) * 190
    plot_df['species_index'] = plot_df['species'].map({species: idx for idx, species in enumerate(SPECIES_ORDER)})
    return plot_df.sort_values(['species_index', 'gene_position']).copy(), display_genes, gene_positions


def draw_peak_query_dotplot_row(
    ax: Axes,
    row_df: pd.DataFrame,
    species: str,
    display_genes: list[str],
    gene_positions: dict[str, float],
    show_headers: bool = False,
) -> None:
    ax.axhspan(-0.5, 0.5, color='#f8f8f8', zorder=0)
    if not row_df.empty:
        ax.scatter(
            row_df['gene_position'],
            np.zeros(len(row_df)),
            s=row_df['size'],
            c=row_df['avg_expr_minmax'],
            cmap='Reds',
            vmin=0.0,
            vmax=1.0,
            edgecolors='#333333',
            linewidths=0.4,
            zorder=2,
        )
    else:
        ax.text(0.0, 0.0, 'NA', ha='center', va='center', fontsize=9, color='#666666')
    x_positions = [gene_positions[gene] for gene in display_genes] if display_genes else [0.0]
    ax.set_xlim(min(x_positions) - 0.12, max(x_positions) + 0.12)
    ax.set_ylim(-0.55, 0.55)
    ax.set_yticks([])
    ax.set_xticks(x_positions)
    if show_headers:
        ax.set_xticklabels(display_genes, rotation=40, ha='left', fontsize=10, fontweight='bold')
        ax.tick_params(axis='x', top=True, labeltop=True, bottom=False, labelbottom=False, length=0, pad=1)
    else:
        ax.set_xticklabels([''] * len(x_positions))
        ax.tick_params(axis='x', top=False, labeltop=False, bottom=False, labelbottom=False, length=0)
    ax.set_ylabel(species, rotation=0, ha='right', va='center', fontsize=10, labelpad=22)
    sns.despine(ax=ax, left=True, right=True, bottom=not show_headers, top=not show_headers)


def format_clean_coordinate_axis(ax: Axes, chrom: str, start: int, end: int) -> None:
    ax.set_xlim(start, end)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4, integer=True, min_n_ticks=3))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{int(round(value)):,}'))
    ax.tick_params(axis='x', labelsize=10, length=0, pad=4)
    ax.xaxis.offsetText.set_visible(False)
    ax.set_xlabel(f'{chrom}:{start:,}-{end:,}', fontsize=11, fontweight='bold', labelpad=6)


def plot_scored_peaks_axis(
    ax: Axes,
    scored_peak_df: pd.DataFrame,
    start: int,
    end: int,
    highlight_peak_id: str | None,
) -> None:
    ax.set_xlim(start, end)
    ax.set_ylim(-0.6, 0.6)
    ax.set_yticks([])
    ax.tick_params(axis='x', bottom=False, labelbottom=False, length=0)
    ax.set_ylabel('Peaks', fontsize=10, rotation=0, ha='right', va='center', labelpad=22)
    sns.despine(ax=ax, top=True, right=True, bottom=True, left=True)
    if scored_peak_df.empty:
        ax.text((start + end) / 2, 0.0, 'No peaks in locus', ha='center', va='center', fontsize=9, color='#666666')
        return
    valid_scores = pd.to_numeric(scored_peak_df['contrast_score'], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    score_limit = max(1.0, float(np.nanquantile(np.abs(valid_scores), 0.98))) if not valid_scores.empty else 1.0
    norm = TwoSlopeNorm(vmin=-score_limit, vcenter=0.0, vmax=score_limit)
    cmap = plt.get_cmap('RdBu_r')
    for _, row in scored_peak_df.iterrows():
        is_highlight = highlight_peak_id is not None and str(row['peak_id']) == str(highlight_peak_id)
        score = row['contrast_score']
        if pd.notna(score):
            facecolor = cmap(norm(float(score)))
        else:
            facecolor = '#bdbdbd'
        rect = Rectangle(
            (float(row['start']), -0.28),
            float(max(1, row['end'] - row['start'])),
            0.56,
            facecolor=facecolor,
            edgecolor='#111111' if is_highlight else 'none',
            linewidth=1.0 if is_highlight else 0.0,
            alpha=0.95,
        )
        ax.add_patch(rect)


def plot_gene_models_with_exons(
    ax: Axes,
    region_gene_coords: pd.DataFrame,
    exon_df: pd.DataFrame,
    start: int,
    end: int,
) -> None:
    ax.set_xlim(start, end)
    if region_gene_coords.empty:
        ax.axis('off')
        return
    gene_order = region_gene_coords['gene_name'].tolist()
    gene_to_y = {gene: idx for idx, gene in enumerate(gene_order[::-1])}
    for _, row in region_gene_coords.iterrows():
        gene_name = row['gene_name']
        y_value = gene_to_y[gene_name]
        ax.plot([int(row['start']), int(row['end'])], [y_value, y_value], color='#4a4a4a', linewidth=1.4, solid_capstyle='round', zorder=1)
        gene_exons = exon_df[(exon_df['gene_name'] == gene_name) & (exon_df['end'] >= start) & (exon_df['start'] <= end)].copy()
        intervals = merge_intervals(sorted((int(exon_row['start']), int(exon_row['end'])) for _, exon_row in gene_exons.iterrows()))
        for exon_start, exon_end in intervals:
            rect = Rectangle((exon_start, y_value - 0.22), exon_end - exon_start, 0.44, facecolor='#1f1f1f', edgecolor='none', zorder=2)
            ax.add_patch(rect)
    ax.set_ylim(-0.6, len(gene_to_y) - 0.4)
    ax.set_yticks(list(gene_to_y.values()))
    ax.set_yticklabels(list(gene_to_y.keys()), fontsize=9)
    sns.despine(ax=ax, top=True, right=True)
    format_clean_coordinate_axis(ax, str(region_gene_coords['chrom'].iloc[0]), start, end)


def plot_scored_peak_panel(
    bw_paths: dict[str, str],
    region: str,
    cell_type: str,
    gene_order: list[str],
    region_gene_coords: pd.DataFrame,
    region_exon_df: pd.DataFrame,
    scored_peak_df: pd.DataFrame,
    out_base: Path,
    highlight_peak_id: str | None = None,
    smooth_window: int = 25,
) -> tuple[Path, Path, pd.DataFrame]:
    dotplot_df = compute_triplet_dotplot_stats(cell_type, gene_order)
    dotplot_display_df, display_genes, gene_positions = prepare_peak_query_dotplot_display(dotplot_df, gene_order)
    chrom, start, end = parse_region(region)
    x = np.arange(start, end)
    signal_map = {species: extract_bigwig_signal(bw_paths[species], chrom, start, end, smooth_window=smooth_window) for species in SPECIES_ORDER}
    max_values = []
    for values in signal_map.values():
        finite_values = np.asarray(values)[np.isfinite(values)]
        max_values.append(float(finite_values.max()) if finite_values.size else 0.0)
    human_max = max_values[SPECIES_ORDER.index('Human')] if 'Human' in SPECIES_ORDER else 0.0
    scale_max = human_max if human_max > 0 else max(max_values)
    scale_max = max(scale_max, 1e-6)

    left_width = max(1.4, 0.34 * max(len(display_genes), 1) + 0.9)
    fig: Figure = plt.figure(figsize=(left_width + 9.8, len(SPECIES_ORDER) * 1.0 + 4.2))
    gs = fig.add_gridspec(
        len(SPECIES_ORDER) + 2,
        2,
        width_ratios=[left_width, 8.6],
        height_ratios=[1.0] * len(SPECIES_ORDER) + [1.1, 0.95],
        hspace=0.16,
        wspace=0.05,
    )

    track_axes = []
    for row_idx, species in enumerate(SPECIES_ORDER):
        ax_dot = fig.add_subplot(gs[row_idx, 0])
        species_df = dotplot_display_df[dotplot_display_df['species'] == species].copy()
        draw_peak_query_dotplot_row(ax_dot, species_df, species, display_genes, gene_positions, show_headers=row_idx == 0)

        ax_track = fig.add_subplot(gs[row_idx, 1], sharex=track_axes[0] if track_axes else None)
        track_axes.append(ax_track)
        scaled = np.clip(np.asarray(signal_map[species], dtype=float) / scale_max, a_min=0.0, a_max=None)
        ax_track.fill_between(x, scaled, color=SPECIES_COLORS[species], alpha=0.92, linewidth=0)
        ax_track.plot(x, scaled, color=SPECIES_COLORS[species], linewidth=0.8)
        ax_track.set_ylim(0, 1.05)
        ax_track.set_yticks([0, 1])
        ax_track.tick_params(axis='y', labelsize=8)
        ax_track.tick_params(axis='x', bottom=False, labelbottom=False, length=0)
        sns.despine(ax=ax_track, top=True, right=True, bottom=True)

    ax_blank = fig.add_subplot(gs[len(SPECIES_ORDER), 0])
    ax_blank.axis('off')
    ax_peaks = fig.add_subplot(gs[len(SPECIES_ORDER), 1], sharex=track_axes[0] if track_axes else None)
    plot_scored_peaks_axis(ax_peaks, scored_peak_df, start, end, highlight_peak_id=highlight_peak_id)

    ax_blank2 = fig.add_subplot(gs[len(SPECIES_ORDER) + 1, 0])
    ax_blank2.axis('off')
    ax_gene = fig.add_subplot(gs[len(SPECIES_ORDER) + 1, 1], sharex=track_axes[0] if track_axes else None)
    plot_gene_models_with_exons(ax_gene, region_gene_coords, region_exon_df, start, end)

    fig.subplots_adjust(left=0.12, right=0.985)
    return save_figure(fig, out_base, tight=False) + (dotplot_display_df,)


peak_annotation = annotation_df.loc[annotation_df['peak_id'] == PEAK_QUERY].copy()
peak_atac_df = atac_results[PEAK_QUERY_CELL_TYPE]
peak_atac_hit = peak_atac_df.loc[peak_atac_df['peak_id'] == PEAK_QUERY].copy()
peak_rna_hits = rna_results[PEAK_QUERY_CELL_TYPE].loc[
    rna_results[PEAK_QUERY_CELL_TYPE]['gene_key'].isin([gene.upper() for gene in PEAK_QUERY_FORCE_GENES])
].copy()

global_links = scenic_triplets.loc[scenic_triplets['peak_id'].astype(str) == PEAK_QUERY].copy()
global_links['scenic_source'] = scenic_source

celltype_links = pd.DataFrame()
if SCENIC_CELLTYPE_PATH.exists():
    scenic_celltype_df = pd.read_csv(SCENIC_CELLTYPE_PATH, sep='\t')
    if 'peak_id' in scenic_celltype_df.columns:
        celltype_links = scenic_celltype_df.loc[scenic_celltype_df['peak_id'].astype(str) == PEAK_QUERY].copy()
        if not celltype_links.empty and 'cell_type' in scenic_celltype_df.columns:
            celltype_links = celltype_links[celltype_links['cell_type'].astype(str) == PEAK_QUERY_CELL_TYPE].copy()
        if not celltype_links.empty:
            celltype_links['scenic_source'] = 'cell_type_specific'

peak_scenic_links = pd.concat([global_links, celltype_links], ignore_index=True, sort=False)
if not peak_scenic_links.empty:
    keep_cols = [column for column in ['scenic_source', 'cell_type', 'TF', 'Gene', 'Region', 'peak_id', 'regulation', 'triplet_rank'] if column in peak_scenic_links.columns]
    peak_scenic_links = peak_scenic_links[keep_cols].drop_duplicates().copy()

linked_tf_genes = peak_scenic_links['TF'].astype(str).tolist() if 'TF' in peak_scenic_links.columns else []
linked_target_genes = peak_scenic_links['Gene'].astype(str).tolist() if 'Gene' in peak_scenic_links.columns else []

oas_gene_coords = load_gene_coordinates(GTF_PATH, PEAK_QUERY_FORCE_GENES)
oas_exon_df = load_gene_exons(GTF_PATH, PEAK_QUERY_FORCE_GENES)
if oas_gene_coords.empty:
    raise ValueError('Could not find OAS gene coordinates in the GTF.')

whole_region, whole_start, whole_end, whole_region_gene_coords = build_panel_region(
    oas_gene_coords,
    PEAK_QUERY_FORCE_GENES,
    flank=PEAK_QUERY_LOCUS_FLANK,
)
whole_chrom, _, _ = parse_region(whole_region)
whole_scored_peak_df = collect_scored_peaks(peak_annotation.append(annotation_df.loc[annotation_df['peak_id'] != PEAK_QUERY], ignore_index=True), peak_atac_df, whole_chrom, whole_start, whole_end)
locus_peak_path = peak_query_dir / f'{PEAK_QUERY}__oas_locus_peaks.tsv'
export_peak_table(whole_scored_peak_df, highlight_peak_id=PEAK_QUERY).to_csv(locus_peak_path, sep='\t', index=False)

oas_gene_coord_path = peak_query_dir / f'{PEAK_QUERY}__oas_gene_coordinates.tsv'
oas_gene_coords.to_csv(oas_gene_coord_path, sep='\t', index=False)

forced_gene_summary = {
    f'{gene.lower()}_log2fc': float(peak_rna_hits.loc[peak_rna_hits['gene_key'] == gene.upper(), 'log2FoldChange'].iloc[0]) if (peak_rna_hits['gene_key'] == gene.upper()).any() else None
    for gene in PEAK_QUERY_FORCE_GENES
}
forced_gene_summary.update({
    f'{gene.lower()}_padj': float(peak_rna_hits.loc[peak_rna_hits['gene_key'] == gene.upper(), 'padj'].iloc[0]) if (peak_rna_hits['gene_key'] == gene.upper()).any() else None
    for gene in PEAK_QUERY_FORCE_GENES
})

peak_query_summary = pd.DataFrame([{
    'peak_id': PEAK_QUERY,
    'cell_type': PEAK_QUERY_CELL_TYPE,
    'region': whole_region,
    'annotation_nearest_gene': None if peak_annotation.empty else peak_annotation['nearest_gene'].dropna().iloc[0] if peak_annotation['nearest_gene'].notna().any() else None,
    'annotation_gene_distance': None if peak_annotation.empty else peak_annotation['gene_distance'].dropna().iloc[0] if peak_annotation['gene_distance'].notna().any() else None,
    'atac_log2fc': None if peak_atac_hit.empty else float(peak_atac_hit['log2FoldChange'].iloc[0]),
    'atac_padj': None if peak_atac_hit.empty else float(peak_atac_hit['padj'].iloc[0]),
    'atac_contrast_score': None if peak_atac_hit.empty else float(compute_atac_contrast_score(peak_atac_hit['log2FoldChange'], peak_atac_hit['padj']).iloc[0]),
    'n_scenic_links': len(peak_scenic_links),
    'n_locus_peaks': len(whole_scored_peak_df),
    **forced_gene_summary,
}])
peak_query_summary_path = peak_query_dir / f'{PEAK_QUERY}__summary.tsv'
peak_query_summary.to_csv(peak_query_summary_path, sep='\t', index=False)

if not peak_scenic_links.empty:
    peak_scenic_path = peak_query_dir / f'{PEAK_QUERY}__scenic_links.tsv'
    peak_scenic_links.to_csv(peak_scenic_path, sep='\t', index=False)
else:
    peak_scenic_path = None

bw_paths, matched_folder = bigwig_bundle_for_cell_type(PEAK_QUERY_CELL_TYPE)
if bw_paths is None:
    raise ValueError(f'No six-species BigWig bundle available for {PEAK_QUERY_CELL_TYPE}.')

panel_records = []
for panel_slug, panel_genes in PEAK_QUERY_PANEL_SPECS:
    panel_region, panel_start, panel_end, panel_region_gene_coords = build_panel_region(oas_gene_coords, panel_genes, flank=PEAK_QUERY_LOCUS_FLANK)
    panel_region_exons = oas_exon_df[oas_exon_df['gene_name'].isin(panel_region_gene_coords['gene_name'])].copy()
    panel_chrom, _, _ = parse_region(panel_region)
    panel_scored_peak_df = collect_scored_peaks(annotation_df, peak_atac_df, panel_chrom, panel_start, panel_end)
    panel_peak_path = peak_query_dir / f'{PEAK_QUERY}__{panel_slug}_peaks.tsv'
    export_peak_table(panel_scored_peak_df, highlight_peak_id=PEAK_QUERY).to_csv(panel_peak_path, sep='\t', index=False)
    gene_order = list(dict.fromkeys(panel_genes + linked_target_genes + linked_tf_genes))
    panel_base = peak_query_dir / f'{PEAK_QUERY}__plasma_b_cells_{panel_slug}_locus_panel'
    panel_pdf, panel_png, peak_dotplot_df = plot_scored_peak_panel(
        bw_paths=bw_paths,
        region=panel_region,
        cell_type=PEAK_QUERY_CELL_TYPE,
        gene_order=gene_order,
        region_gene_coords=panel_region_gene_coords,
        region_exon_df=panel_region_exons,
        scored_peak_df=panel_scored_peak_df,
        out_base=panel_base,
        highlight_peak_id=PEAK_QUERY,
    )
    peak_dotplot_path = panel_base.with_suffix('.tsv')
    peak_dotplot_df.to_csv(peak_dotplot_path, sep='\t', index=False)
    panel_records.append({
        'panel': panel_slug,
        'genes': ';'.join(panel_genes),
        'region': panel_region,
        'n_region_genes': len(panel_region_gene_coords),
        'n_peaks': len(panel_scored_peak_df),
        'peaks_tsv': str(panel_peak_path),
        'pdf_path': str(panel_pdf),
        'png_path': str(panel_png),
        'dotplot_tsv': str(peak_dotplot_path),
    })

peak_panel_manifest_df = pd.DataFrame(panel_records)
peak_panel_manifest_path = peak_query_dir / f'{PEAK_QUERY}__panel_manifest.tsv'
peak_panel_manifest_df.to_csv(peak_panel_manifest_path, sep='\t', index=False)

triplet_target_upper = [gene.upper() for gene in TRIPLET_TARGET_GENES]
triplet_candidates = ranked_triplet_df[
    ranked_triplet_df['Gene'].astype(str).str.upper().isin(triplet_target_upper)
].copy()
triplet_target_summary_rows = []
triplet_panel_records = []
if not triplet_candidates.empty:
    target_order = {gene: idx for idx, gene in enumerate(triplet_target_upper)}
    triplet_candidates['target_order'] = triplet_candidates['Gene'].astype(str).str.upper().map(target_order).fillna(len(target_order))
    triplet_candidates = triplet_candidates.sort_values(
        ['target_order', 'cell_type', 'direction', 'core_sig', 'triplet_sig', 'total_score_abs', 'sig_metric', 'triplet_rank'],
        ascending=[True, True, True, False, False, False, False, True],
    ).reset_index(drop=True)
    target_gene_coords = load_gene_coordinates(GTF_PATH, sorted(triplet_candidates['Gene'].astype(str).unique()))

    for target_gene_upper, target_df in triplet_candidates.groupby(triplet_candidates['Gene'].astype(str).str.upper(), sort=False):
        target_gene_name = str(target_df['Gene'].iloc[0])
        target_dir = triplet_locus_dir / output_slug(target_gene_name)
        target_dir.mkdir(parents=True, exist_ok=True)
        rendered_count = 0
        for rank_idx, (_, row) in enumerate(target_df.iterrows(), start=1):
            cell_type = str(row['cell_type'])
            bw_paths, matched_folder = bigwig_bundle_for_cell_type(cell_type)
            if bw_paths is None:
                continue
            target_coords = target_gene_coords[target_gene_coords['gene_name'].astype(str).str.upper() == target_gene_upper].copy()
            if target_coords.empty:
                continue
            triplet_region, triplet_start, triplet_end = build_target_peak_region(
                annotation_df,
                target_coords,
                str(row['peak_id']),
                flank=TRIPLET_LOCUS_FLANK,
            )
            triplet_chrom, _, _ = parse_region(triplet_region)
            region_gene_coords, region_exon_df = load_region_gene_models(GTF_PATH, triplet_chrom, triplet_start, triplet_end)
            if region_gene_coords.empty:
                region_gene_coords = target_coords.copy()
                region_exon_df = load_gene_exons(GTF_PATH, target_coords['gene_name'].tolist())
            scored_peak_df = collect_scored_peaks(annotation_df, atac_results[cell_type], triplet_chrom, triplet_start, triplet_end)
            panel_base = target_dir / (
                f'{rank_idx:03d}_{output_slug(cell_type)}__{sanitize_token(str(row["TF"]))}__{sanitize_token(target_gene_name)}__{sanitize_token(str(row["peak_id"]))}'
            )
            panel_pdf, panel_png, triplet_dotplot_df = plot_scored_peak_panel(
                bw_paths=bw_paths,
                region=triplet_region,
                cell_type=cell_type,
                gene_order=[target_gene_name, str(row['TF'])],
                region_gene_coords=region_gene_coords,
                region_exon_df=region_exon_df,
                scored_peak_df=scored_peak_df,
                out_base=panel_base,
                highlight_peak_id=str(row['peak_id']),
            )
            dotplot_path = panel_base.with_suffix('.tsv')
            triplet_dotplot_df.to_csv(dotplot_path, sep='\t', index=False)
            peaks_path = panel_base.parent / f'{panel_base.name}__peaks.tsv'
            export_peak_table(scored_peak_df, highlight_peak_id=str(row['peak_id'])).to_csv(peaks_path, sep='\t', index=False)
            triplet_panel_records.append({
                'target_gene': target_gene_name,
                'cell_type': cell_type,
                'bigwig_folder': matched_folder,
                'TF': str(row['TF']),
                'Gene': target_gene_name,
                'peak_id': str(row['peak_id']),
                'peak_region': peak_to_region(annotation_df, str(row['peak_id'])),
                'locus_region': triplet_region,
                'direction': str(row['direction']),
                'core_sig': bool(row['core_sig']),
                'triplet_sig': bool(row['triplet_sig']),
                'da_score': float(row['da_score']),
                'tf_score': float(row['tf_score']),
                'target_score': float(row['target_score']),
                'total_score': float(row['total_score']),
                'pdf_path': str(panel_pdf),
                'png_path': str(panel_png),
                'dotplot_tsv': str(dotplot_path),
                'peaks_tsv': str(peaks_path),
            })
            rendered_count += 1
        triplet_target_summary_rows.append({
            'target_gene': target_gene_name,
            'n_triplets': len(target_df),
            'n_rendered': rendered_count,
            'output_dir': str(target_dir),
        })

triplet_target_summary_df = pd.DataFrame(triplet_target_summary_rows)
triplet_target_summary_path = triplet_locus_dir / 'triplet_target_summary.tsv'
triplet_target_summary_df.to_csv(triplet_target_summary_path, sep='\t', index=False)

triplet_panel_manifest_df = pd.DataFrame(triplet_panel_records)
triplet_panel_manifest_path = triplet_locus_dir / 'triplet_locus_panel_manifest.tsv'
triplet_panel_manifest_df.to_csv(triplet_panel_manifest_path, sep='\t', index=False)

display(peak_query_summary)
display(peak_scenic_links if not peak_scenic_links.empty else pd.DataFrame({'note': ['No direct SCENIC+ links found for unified_244384 in the current tables.']}))
display(oas_gene_coords)
display(peak_panel_manifest_df)
display(triplet_target_summary_df if not triplet_target_summary_df.empty else pd.DataFrame({'note': ['No ranked triplets found for the requested target genes.']}))
display(triplet_panel_manifest_df.head(20) if not triplet_panel_manifest_df.empty else pd.DataFrame({'note': ['No triplet locus panels were rendered.']}))
print('Saved peak query summary:', peak_query_summary_path)
print('Saved OAS gene coordinates:', oas_gene_coord_path)
print('Saved OAS locus peaks:', locus_peak_path)
print('Saved peak query SCENIC links:', peak_scenic_path if peak_scenic_path is not None else 'no direct links found')
print('Saved peak query panel manifest:', peak_panel_manifest_path)
print('Saved triplet target summary:', triplet_target_summary_path)
print('Saved triplet locus panel manifest:', triplet_panel_manifest_path)

,peak_id,cell_type,region,annotation_nearest_gene,annotation_gene_distance,atac_log2fc,atac_padj,atac_contrast_score,n_scenic_links,n_locus_peaks,oas1_log2fc,oas2_log2fc,oas3_log2fc,oas1_padj,oas2_padj,oas3_padj
0,unified_244384,Plasma B cells,chr12:112895856-113021723,IMMP1LP2,16149.0,3.195427,1.349900e-12,37.928758,0,69,0.235074,-0.356552,-1.082961,0.852061,0.251706,0.203049


,note
0,No direct SCENIC+ links found for unified_2443...


,gene_name,chrom,start,end
0,OAS1,chr12,112905856,112933219
1,OAS3,chr12,112938051,112976460
2,OAS2,chr12,112978395,113011723


,panel,genes,region,n_region_genes,n_peaks,peaks_tsv,pdf_path,png_path,dotplot_tsv
0,oas1,OAS1,chr12:112895856-112943219,2,47,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
1,oas1_oas2,OAS1;OAS2,chr12:112895856-113021723,3,69,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
2,oas_all,OAS1;OAS2;OAS3,chr12:112895856-113021723,3,69,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...


,target_gene,n_triplets,n_rendered,output_dir
0,AREG,15,15,/home/jjanssens/jjans/analysis/adult_intestine...
1,ALPI,1,1,/home/jjanssens/jjans/analysis/adult_intestine...


,target_gene,cell_type,bigwig_folder,TF,Gene,peak_id,peak_region,locus_region,direction,core_sig,triplet_sig,da_score,tf_score,target_score,total_score,pdf_path,png_path,dotplot_tsv,peaks_tsv
0,AREG,Enterocytes,enterocytes,KLF4,AREG,unified_731237,chr4:74415273-74421773,chr4:74408273-74465005,Human up,False,False,9.192357,1.733402,0.391886,11.317645,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
1,AREG,Enterocytes,enterocytes,KLF6,AREG,unified_731237,chr4:74415273-74421773,chr4:74408273-74465005,Human up,False,False,9.192357,0.572185,0.391886,10.156428,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
2,AREG,Enterocytes,enterocytes,ATF3,AREG,unified_731237,chr4:74415273-74421773,chr4:74408273-74465005,Human up,False,False,9.192357,0.309705,0.391886,9.893948,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
3,AREG,Enterocytes,enterocytes,KLF4,AREG,unified_731251,chr4:74441868-74448368,chr4:74434868-74465005,Human up,False,False,1.503388,1.733402,0.391886,3.628677,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
4,AREG,Enterocytes,enterocytes,KLF6,AREG,unified_731251,chr4:74441868-74448368,chr4:74434868-74465005,Human up,False,False,1.503388,0.572185,0.391886,2.467459,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
5,AREG,Enterocytes,enterocytes,ATF3,AREG,unified_731251,chr4:74441868-74448368,chr4:74434868-74465005,Human up,False,False,1.503388,0.309705,0.391886,2.204980,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
6,AREG,Macrophages,macrophages,KLF4,AREG,unified_731251,chr4:74441868-74448368,chr4:74434868-74465005,Human up,True,False,2.008022,0.477320,5.108417,7.593759,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
7,AREG,Macrophages,macrophages,KLF6,AREG,unified_731251,chr4:74441868-74448368,chr4:74434868-74465005,Human up,True,False,2.008022,0.001271,5.108417,7.117711,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
8,AREG,Naive B cells,naive_b_cells,KLF4,AREG,unified_731251,chr4:74441868-74448368,chr4:74434868-74465005,Human up,False,False,9.459356,12.316984,1.247734,23.024073,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
9,AREG,Naive B cells,naive_b_cells,ATF3,AREG,unified_731251,chr4:74441868-74448368,chr4:74434868-74465005,Human up,False,False,9.459356,1.384388,1.247734,12.091477,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...


Saved peak query summary: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/peak_queries/plasma_b_cells/unified_244384__summary.tsv
Saved OAS gene coordinates: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/peak_queries/plasma_b_cells/unified_244384__oas_gene_coordinates.tsv
Saved OAS locus peaks: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/peak_queries/plasma_b_cells/unified_244384__oas_locus_peaks.tsv
Saved peak query SCENIC links: no direct links found
Saved peak query panel manifest: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulatory_panels/peak_queries/plasma_b_cells/unified_244384__panel_manifest.tsv
Saved triplet target summary: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/22_publication_regulato

## Outputs



Key outputs written by this notebook:

- `figures/volcano_da/<cell_type>__da_volcano_human_vs_rest.{pdf,png}`

- `figures/volcano_de/<cell_type>__de_volcano_human_vs_rest.{pdf,png}`

- `figures/triplet_scatter/<cell_type>__triplet_scatter.{pdf,png}`

- `tables/all_ranked_triplets.tsv`

- `tables/<cell_type>__selected_top10_triplets.tsv`

- `tables/triplet_track_manifest.tsv`

- `tables/selected_triplet_expression_manifest.tsv`

- `tables/gene_sets/gene_set_triplet_summary.tsv`

- `tables/gene_sets/<cell_type>__<gene_set>__triplets.tsv`

- `triplet_tracks/<cell_type>/*.{pdf,png}`

- `gene_set_triplet_figures/<cell_type>/<gene_set>__triplet_scatter.{pdf,png}`

- `gene_set_expression_heatmaps/<cell_type>/selected_triplet_genes_expression.{pdf,png,tsv}`

- `gene_set_expression_heatmaps/gene_sets/<cell_type>/<gene_set>__target_expression.{pdf,png,tsv}`



Triplet highlight selection is per cell type and prefers enhancer-target pairs that are both significantly differential in that same cell type.